# TrunCat: ClinVar example — VCF to NMD predictions

Run cells **from top to bottom** in Google Colab. This workflow starts from the public example VCF, installs dependencies, constructs annotations and features, predicts with TrunCat, extracts ClinVar significance, and plots the predictions.

**Execution status:** assembled from the project workflow; a clean Colab `Run all` has not been verified. Some upstream scripts require reference data and corrections that must be published in the repository. Cells fail visibly when inputs are absent instead of substituting a precomputed 619-feature matrix.

## 1. Clone the GitHub repository and load the example ClinVar VCF

In [ ]:
from pathlib import Path
import subprocess, os, sys, shutil
ROOT = Path('/content/NMDpredictionmodel')
if not (ROOT / '.git').exists():
    subprocess.run(['git','clone','--quiet','https://github.com/CobanAkdemirlab/NMDpredictionmodel.git',str(ROOT)],check=True)
VCF = ROOT / 'examples/clinvar_20260201_sample5k.vcf.gz'
assert VCF.is_file(), f'Missing GitHub example VCF: {VCF}'
print('Example VCF:', VCF)
print('Repository:', ROOT)


## 2. Dependencies and reference files

In [ ]:
setup = r'''
options(repos=c(CRAN="https://cloud.r-project.org"))
if (!requireNamespace("BiocManager",quietly=TRUE)) install.packages("BiocManager")
for (p in c("GenomicRanges","VariantAnnotation","rtracklayer","GenomicFeatures","AnnotationDbi")) {
  if (!requireNamespace(p,quietly=TRUE)) BiocManager::install(p,ask=FALSE,update=FALSE)
}
for (p in c("aenmd","aenmd.data.ensdb.v105")) {
  cat(p,requireNamespace(p,quietly=TRUE),"\n")
}
'''
r=subprocess.run(['Rscript','-e',setup],text=True)
assert r.returncode==0,'R package setup failed'


In [ ]:
import subprocess

r_code = r'''
options(repos = c(CRAN = "https://cloud.r-project.org"))

if (!requireNamespace("remotes", quietly = TRUE)) {
    install.packages("remotes")
}

if (!requireNamespace("aenmd.data.ensdb.v105", quietly = TRUE)) {
    remotes::install_github(
        "kostkalab/aenmd_data",
        subdir = "aenmd.data.ensdb.v105",
        dependencies = TRUE,
        upgrade = "never"
    )
}

if (!requireNamespace("aenmd", quietly = TRUE)) {
    remotes::install_github(
        "kostkalab/aenmd",
        dependencies = TRUE,
        upgrade = "never"
    )
}

cat("aenmd:", requireNamespace("aenmd", quietly = TRUE), "\n")
cat("ensdb:", requireNamespace("aenmd.data.ensdb.v105", quietly = TRUE), "\n")
'''

result = subprocess.run(
    ["Rscript", "-e", r_code],
    capture_output=True,
    text=True
)

print(result.stdout[-3000:])
print(result.stderr[-3000:])
print("Exit code:", result.returncode)

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','pandas','numpy','catboost','gffutils','pyfaidx','pyranges','requests','biopython','pysam','matplotlib'],check=True)
for sub in ['Annotation/reference','Annotation/output','Features/reference','Features/output','Model/TrunCat/data','Model/TrunCat/predictions']:
    (ROOT/sub).mkdir(parents=True,exist_ok=True)


## 3. ClinVar PTC extraction and canonical annotation

In [ ]:
from pathlib import Path
import subprocess

original = ROOT / "Dataset_extraction/03_ClinVar_extraction.R"

# Read the original script
code = original.read_text()

# Change only the input/output configuration for Colab
code = code.replace(
    'input_dir   = "~/ClinVar2026"',
    'input_dir   = "/content/NMDpredictionmodel/examples"'
)

code = code.replace(
    'output_dir  = "~/ClinVar2026/output"',
    'output_dir  = "/content/NMDpredictionmodel/data/ClinVar"'
)

code = code.replace(
    'vcf_file    = "clinvar_20260201.vcf.gz"',
    'vcf_file    = "clinvar_20260201_sample5k.vcf.gz"'
)

# Stop before the ANNOVAR preparation section.
# We already have the ANNOVAR output in GitHub.
marker = '# 9. Prepare VCF-like input for ANNOVAR'
assert marker in code, "Could not locate ANNOVAR section"

code = code[:code.index(marker)]

# Remove the section separator immediately before the marker
# (does not affect R execution).
script = Path("/content/clinvar_original_sample5k.R")
script.write_text(code)

print("Using original ClinVar extraction logic")
print("Script:", script)

# Execute only if the PTC checkpoint is missing
if not (ROOT / PTC).exists():
    log = Path("/content/clinvar_extraction.log")

    with open(log, "w") as f:
        result = subprocess.run(
            ["Rscript", "--vanilla", str(script)],
            stdout=f,
            stderr=subprocess.STDOUT
        )

    print("\n=== EXTRACTION LOG ===")
    print("\n".join(log.read_text(errors="replace").splitlines()[-35:]))
    print("Exit code:", result.returncode)

    if result.returncode != 0:
        raise RuntimeError("Original ClinVar extraction failed; see log.")
else:
    print("Existing PTC checkpoint found. Skipping extraction.")

# Verify and immediately back up the output
assert (ROOT / PTC).exists(), "PTC RDS was not created"
save(PTC)

print("\nSUCCESS: ClinVar PTC SNV checkpoint saved to Google Drive")

In [ ]:
from pathlib import Path
import subprocess
import shutil

# Original Step 01 script
original = ROOT / "Annotation/01_variant_annotation.R"
code = original.read_text()

# 1. Set the Colab project directory
old_path = 'PROJECT_DIR <- "/path/to/NMDpredictionmodel"'
assert old_path in code, "PROJECT_DIR placeholder not found"

code = code.replace(
    old_path,
    f'PROJECT_DIR <- "{ROOT}"',
    1
)

# 2. Run ClinVar only, keeping the original processing functions
# Replace the DATASETS block with the existing ClinVar configuration.
start = code.index("DATASETS <- list(")
end = code.index("\n\n# ", start)

clinvar_config = '''
DATASETS <- list(
    ClinVar = list(
        annovar_file = file.path(
            PROJECT_DIR, "data", "ClinVar",
            "clinvar_20260201_PTC_SNV_ensGene.exonic_variant_function"
        ),
        variant_file = file.path(
            PROJECT_DIR, "data", "ClinVar",
            "clinvar_20260201_PTC_SNV.rds"
        )
    )
)
'''

# Locate the complete DATASETS block safely by its final closing brace
# immediately before the next section.
code = code[:start] + clinvar_config + code[end:]

# 3. Copy precomputed ANNOVAR annotation from GitHub examples
source = ROOT / "examples/clinvar_sample5k_PTC_SNV_ensGene.exonic_variant_function"
destination = ROOT / ANN

assert source.exists(), f"Missing ANNOVAR file: {source}"
assert (ROOT / PTC).exists(), "Missing ClinVar PTC RDS"

destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(source, destination)

# 4. Save temporary Colab script
script = Path("/content/clinvar_step01.R")
script.write_text(code)

# 5. Run Step 01
log = Path("/content/clinvar_step01.log")

with open(log, "w") as f:
    result = subprocess.run(
        ["Rscript", "--vanilla", str(script)],
        stdout=f,
        stderr=subprocess.STDOUT
    )

print("=== STEP 01 LOG ===")
print("\n".join(log.read_text(errors="replace").splitlines()[-45:]))
print("Exit code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError("Step 01 failed. See log above.")

# 6. Verify and back up canonical annotation
assert (ROOT / CAN).exists(), "Canonical annotation RDS not created"
save(CAN)

print("\nSUCCESS: ClinVar canonical annotation saved to Google Drive")

## 4. GENCODE transcript annotation and PTC geometry

In [ ]:
GTF = 'Annotation/reference/gencode.v26.primary_assembly.annotation.gtf.gz'
CAN = 'Annotation/output/ClinVar_canonical_variant_annotation.rds'
PTC = 'Dataset_extraction/output/ClinVar_PTC_SNV.rds'
ANN = 'Annotation/reference/clinvar_sample5k_PTC_SNV_ensGene.exonic_variant_function'

In [ ]:
from pathlib import Path
import subprocess

gtf = ROOT / GTF

url = (
    "https://ftp.ebi.ac.uk/pub/databases/gencode/"
    "Gencode_human/release_26/"
    "gencode.v26.primary_assembly.annotation.gtf.gz"
)

if not gtf.exists():
    gtf.parent.mkdir(parents=True, exist_ok=True)

    print("Downloading GENCODE v26...")
    subprocess.run(
        ["wget", "-q", "--show-progress", "-O", str(gtf), url],
        check=True
    )
else:
    print("GENCODE v26 already exists. Skipping download.")

print("GENCODE file:", gtf)
print("File size (MB):", round(gtf.stat().st_size / 1e6, 2))

# Save the downloaded reference to Google Drive
save(GTF)

In [ ]:
from pathlib import Path
import subprocess

script = Path("/content/clinvar_step02.R")
original = ROOT / "Annotation/02_GENCODEv26_transcript_structure.R"

filtered_gtf = ROOT / "Annotation/reference/clinvar20_gencode_v26.gtf"
output = ROOT / "Annotation/output/ClinVar_gencode_v26_transcript_structure.rds"

r_code = f'''
library(data.table)

variants <- readRDS("{ROOT / CAN}")
targets <- unique(as.character(variants$txnames))
target_base <- sub("\\\\.[0-9]+$", "", targets)

gtf <- fread(
    cmd = paste("gzip -dc", shQuote("{ROOT / GTF}")),
    sep = "\\t",
    header = FALSE,
    skip = "#",
    showProgress = FALSE
)

setnames(gtf, paste0("V", 1:9))

gtf[, transcript_id := sub(
    '.*transcript_id "([^"]+)".*',
    '\\\\1', V9
)]

gtf[, base_id := sub("\\\\.[0-9]+$", "", transcript_id)]

subset_gtf <- gtf[
    base_id %in% target_base &
    V1 %in% paste0("chr", 1:22)
]

report <- data.table(
    txnames = targets,
    base_id = target_base
)

report[, gtf_records := vapply(
    base_id,
    function(id) sum(subset_gtf$base_id == id),
    integer(1)
)]

report[, cds_segments := vapply(
    base_id,
    function(id) sum(
        subset_gtf$base_id == id &
        subset_gtf$V3 == "CDS"
    ),
    integer(1)
)]

print(report)

cat("\\nTranscripts with CDS:", sum(report$cds_segments > 0), "\\n")
cat("Transcripts without CDS:", sum(report$cds_segments == 0), "\\n")

# Preserve the original GTF's nine-column structure.
fwrite(
    subset_gtf[, paste0("V", 1:9), with = FALSE],
    "{filtered_gtf}",
    sep = "\\t",
    col.names = FALSE,
    quote = FALSE
)
'''

prep = Path("/content/prepare_clinvar_gtf.R")
prep.write_text(r_code)

result = subprocess.run(
    ["Rscript", "--vanilla", str(prep)],
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr[-3000:])
    raise RuntimeError("GTF preparation failed.")

# Build a clean temporary script from the original.
code = original.read_text()

code = code.replace(
    '"/path/to/gencode.v26.primary_assembly.annotation.gtf.gz"',
    f'"{filtered_gtf}"'
)

code = code.replace(
    '"/path/to/gencode_v26_transcript_structure.rds"',
    f'"{output}"'
)

code = code.replace(
    "makeTxDbFromGFF(",
    "txdbmaker::makeTxDbFromGFF("
)

code = code.replace(
    "txdb <- keepSeqlevels(",
    "txdb <- GenomeInfoDb::keepSeqlevels("
)

script.write_text(code)

print("\nPrepared filtered GTF:", filtered_gtf)
print("Prepared original-based Step 02 script:", script)
print("No transcript database was rebuilt.")

In [ ]:
from pathlib import Path
import subprocess

script = Path("/content/clinvar_step02.R")
output = ROOT / "Annotation/output/ClinVar_gencode_v26_transcript_structure.rds"
log = Path("/content/clinvar_step02.log")

code = script.read_text()

old = '''txdb <- GenomeInfoDb::keepSeqlevels(
    txdb,
    chr_list,
    pruning.mode = "coarse"
)'''

new = '''# Retain only autosomes that exist in the filtered ClinVar GTF.
available_chr <- intersect(
    chr_list,
    GenomeInfoDb::seqlevels(txdb)
)

txdb <- GenomeInfoDb::keepSeqlevels(
    txdb,
    available_chr,
    pruning.mode = "coarse"
)'''

if old in code:
    code = code.replace(old, new, 1)
    script.write_text(code)
    print("Fixed chromosome filtering.")
elif new in code:
    print("Chromosome filtering already fixed.")
else:
    raise RuntimeError("Expected chromosome filtering block not found.")

# Remove any previous partial output.
if output.exists():
    output.unlink()

# Run Step 02.
with open(log, "w") as f:
    result = subprocess.run(
        ["Rscript", "--vanilla", str(script)],
        stdout=f,
        stderr=subprocess.STDOUT
    )

print("\n=== STEP 02 LOG ===")
print("\n".join(log.read_text(errors="replace").splitlines()[-25:]))
print("Exit code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError("Step 02 failed. Earlier checkpoints remain safe.")

# Validate the output.
validation_code = f'''
x <- readRDS("{output}")
v <- readRDS("{ROOT / CAN}")

base <- function(z) sub("\\\\.[0-9]+$", "", z)

required <- c(
    "txnames", "cds_length", "cds_exons",
    "exon_count", "noncoding.length", "NCexonsnum"
)

stopifnot(all(required %in% names(x)))

matched <- unique(base(x$txnames))
requested <- unique(base(v$txnames))

cat("Transcript structure rows:", nrow(x), "\\n")
cat("Canonical transcripts matched:",
    sum(requested %in% matched), "/", length(requested), "\\n")

cat("Unmatched transcripts:\\n")
print(setdiff(requested, matched))

stopifnot(
    nrow(x) == 19,
    length(matched) == 19,
    all(is.finite(x$cds_length)),
    all(x$cds_length > 0)
)
'''

check = subprocess.run(
    ["Rscript", "--vanilla", "-e", validation_code],
    capture_output=True,
    text=True
)

print("\n=== VALIDATION ===")
print(check.stdout)

if check.returncode != 0:
    print(check.stderr)
    raise RuntimeError("Validation failed. Output not backed up.")

save(str(output.relative_to(ROOT)))

print("\nSUCCESS: ClinVar Step 02 backed up to Google Drive.")

In [ ]:
from pathlib import Path
import subprocess
import re

original = ROOT / "Annotation/03_PTC_annotation.R"
script = Path("/content/clinvar_step03.R")
output = ROOT / "Annotation/output/ClinVar_PTC_annotated.rds"
log = Path("/content/clinvar_step03.log")

code = original.read_text()

# 1. Set the three ClinVar-specific paths.
paths = {
    "/path/to/TOPMed_canonical_variant_annotation.rds":
        str(ROOT / CAN),
    "/path/to/gencode_v26_transcript_structure.rds":
        str(ROOT / "Annotation/output/ClinVar_gencode_v26_transcript_structure.rds"),
    "/path/to/TOPMed_PTC_annotated.rds":
        str(output)
}

for old, new in paths.items():
    if code.count(old) != 1:
        raise RuntimeError(f"Expected exactly one occurrence of {old}")
    code = code.replace(old, new)

# 2. Replace the exact transcript-version join with a versionless join.
old_join = '''variants <- variants_canonical %>%
    left_join(
        transcript_structure,
        by = "txnames"
    )'''

new_join = '''# Preserve original ClinVar and GENCODE transcript versions.
variants_canonical$txnames_clinvar <- variants_canonical$txnames
transcript_structure$txnames_gencode <- transcript_structure$txnames

base_id <- function(x) sub("\\\\.[0-9]+$", "", x)

variants_canonical$tx_base <- base_id(variants_canonical$txnames)
transcript_structure$tx_base <- base_id(transcript_structure$txnames)

stopifnot(!anyDuplicated(transcript_structure$tx_base))

transcript_structure$txnames <- NULL

variants <- variants_canonical %>%
    left_join(
        transcript_structure,
        by = "tx_base",
        relationship = "many-to-one"
    )'''

if code.count(old_join) != 1:
    raise RuntimeError(
        "Original transcript join not found. "
        "No script was written."
    )

code = code.replace(old_join, new_join, 1)

# 3. Add QC before the original saveRDS call.
qc = '''
# ClinVar-specific validation before saving.
stopifnot(
    nrow(variants_annotated) == 20,
    all(!is.na(variants_annotated$coding.pos)),
    all(!is.na(variants_annotated$mut.exon)),
    all(!is.na(variants_annotated$PTC.2.EJC)),
    all(variants_annotated$PTC.2.EJC >= 0),
    all(variants_annotated$coding.pos >= 1),
    all(variants_annotated$coding.pos <= variants_annotated$cds_length),
    all(variants_annotated$mut.exon >= 1),
    all(variants_annotated$mut.exon <= variants_annotated$exon_count),
    all(variants_annotated$EJC.downstream >= 0)
)

message("ClinVar Step 03 QC passed.")
'''

save_marker = '''saveRDS(
    variants_annotated,
    CONFIG$output_file
)'''

if code.count(save_marker) != 1:
    raise RuntimeError("Original saveRDS block not found.")

code = code.replace(save_marker, qc + "\n" + save_marker, 1)

script.write_text(code)

# 4. Run without overwriting an existing checkpoint.
if output.exists():
    output.unlink()

with open(log, "w") as f:
    result = subprocess.run(
        ["Rscript", "--vanilla", str(script)],
        stdout=f,
        stderr=subprocess.STDOUT
    )

print("=== STEP 03 LOG ===")
print("\n".join(log.read_text(errors="replace").splitlines()[-45:]))
print("Exit code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Step 03 failed. Previous Google Drive checkpoints are unchanged."
    )

# 5. Back up the validated output.
save(str(output.relative_to(ROOT)))

print("\nSUCCESS: ClinVar Step 03 backed up to Google Drive.")

In [ ]:
from pathlib import Path
import subprocess
import shutil
import re

# -------------------------------------------------------
# 1. Paths
# -------------------------------------------------------

ROOT = Path("/content/NMDpredictionmodel")
BACKUP = Path("/content/drive/MyDrive/NMD_ClinVar_demo_checkpoints")

original = ROOT / "Annotation/04_shared_variant_filtering.R"
script = Path("/content/clinvar_step04.R")

input_file = ROOT / "Annotation/output/ClinVar_PTC_annotated.rds"
output_file = ROOT / "Annotation/output/ClinVar_PTC_filtered.rds"

expression_file = ROOT / (
    "Annotation/reference/"
    "GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_median_tpm.gct.txt"
)

egenes_file = ROOT / "Annotation/reference/Whole_Blood.v8.egenes.txt"

for p in [original, input_file, expression_file, egenes_file]:
    if not p.is_file():
        raise FileNotFoundError(f"Missing required file: {p}")

# -------------------------------------------------------
# 2. Read original script
# -------------------------------------------------------

code = original.read_text()

# Correct the original configuration.
config_replacements = {
    'CONFIG <- list(': (
        'CONFIG <- list(\n'
        f'    input_file = "{input_file}",'
    ),
    '/path/to/GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_median_tpm.gct':
        str(expression_file),
    '/path/to/Whole_Blood.v8.egenes.txt':
        str(egenes_file),
    '/path/to/PTC_filtered_variants.rds':
        str(output_file),
}

for old, new in config_replacements.items():
    count = code.count(old)
    if count != 1:
        raise RuntimeError(
            f"Expected exactly one occurrence of {old!r}, found {count}"
        )
    code = code.replace(old, new, 1)

# -------------------------------------------------------
# 3. Add input checks and variant ID normalization
# -------------------------------------------------------

# GTEx uses chr1_123_A_G_b38; remove _b38 as in original.
# Also normalize optional 'chr' prefixes on both sides.
old_eqtl = '''eqtl_variant_ids <- unique(
    significant_egenes$variantID
)'''

new_eqtl = '''normalize_variant_id <- function(x) {
    x <- as.character(x)
    x <- sub("_b38$", "", x)
    x <- sub("^chr", "", x, ignore.case = TRUE)
    x
}

eqtl_variant_ids <- unique(
    normalize_variant_id(significant_egenes$variantID)
)

variants$variantID_normalized <- normalize_variant_id(
    variants$variantID
)'''

if code.count(old_eqtl) != 1:
    raise RuntimeError("Could not locate the original eQTL ID block.")

code = code.replace(old_eqtl, new_eqtl, 1)

old_filter = '''!variantID %in%
            eqtl_variant_ids'''

new_filter = '''!variantID_normalized %in%
            eqtl_variant_ids'''

if code.count(old_filter) != 1:
    raise RuntimeError("Could not locate the original eQTL filter.")

code = code.replace(old_filter, new_filter, 1)

# Remove the temporary normalized ID column before saving.
old_final = 'variants_filtered <- variants'

new_final = '''variants_filtered <- variants %>%
    dplyr::select(-variantID_normalized)

stopifnot(
    nrow(variants_filtered) <= 20,
    all(variants_filtered$exon_count > 1),
    !anyNA(variants_filtered$exon_count),
    all(c("key", "variantID", "GENE_ID") %in%
        names(variants_filtered))
)

message("ClinVar Step 04 QC passed.")'''

if code.count(old_final) != 1:
    raise RuntimeError("Could not locate the final output assignment.")

code = code.replace(old_final, new_final, 1)

# -------------------------------------------------------
# 4. Validate the R script before execution
# -------------------------------------------------------

script.write_text(code)

parse_result = subprocess.run(
    [
        "Rscript", "-e",
        f"parse(file={str(script)!r}); cat('R syntax OK\\n')"
    ],
    capture_output=True,
    text=True
)

if parse_result.returncode != 0:
    print(parse_result.stderr)
    raise RuntimeError("Step 04 R syntax validation failed.")

print("R syntax validation passed.")

# -------------------------------------------------------
# 5. Run Step 04
# -------------------------------------------------------

log = Path("/content/clinvar_step04.log")

with log.open("w") as f:
    result = subprocess.run(
        ["Rscript", "--vanilla", str(script)],
        stdout=f,
        stderr=subprocess.STDOUT
    )

print("\n=== STEP 04 LOG ===")
print(log.read_text(errors="replace")[-12000:])
print("Exit code:", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Step 04 failed. Previous checkpoints are unchanged."
    )

if not output_file.is_file():
    raise RuntimeError("Step 04 finished without producing its output.")

# -------------------------------------------------------
# 6. Back up successful output to Google Drive
# -------------------------------------------------------

relative = output_file.relative_to(ROOT)
destination = BACKUP / relative

destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(output_file, destination)

print("\nSUCCESS: Step 04 completed.")
print("Output:", output_file)
print("Google Drive backup:", destination)

## 5. Genomic and transcriptomic feature generation

In [ ]:
from pathlib import Path
import subprocess
import shutil
import os

ROOT = Path("/content/NMDpredictionmodel")
BACKUP = Path("/content/drive/MyDrive/NMD_ClinVar_demo_checkpoints")

SCRIPT = ROOT / "Features/01a_GENCODEv26_sequence_features.R"
GTF = ROOT / "Annotation/reference/clinvar20_gencode_v26.gtf"
OUTPUT = ROOT / "Features/output/ClinVar_gencode_v26_sequence_features.rds"

assert SCRIPT.is_file(), f"Missing script: {SCRIPT}"
assert GTF.is_file(), f"Missing GTF: {GTF}"

OUTPUT.parent.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["GENCODE_GTF"] = str(GTF)
env["SEQUENCE_FEATURES_OUTPUT"] = str(OUTPUT)

print("Running sequence features from GitHub...")
print("Input:", GTF)

result = subprocess.run(
    ["Rscript", str(SCRIPT)],
    cwd=str(ROOT),
    env=env,
    capture_output=True,
    text=True
)

print(result.stdout[-10000:])
print(result.stderr[-10000:])

if result.returncode != 0:
    raise RuntimeError("Sequence feature generation failed")

assert OUTPUT.is_file(), "Expected RDS output was not created"

# Back up output to Google Drive
DEST = BACKUP / "Features/output" / OUTPUT.name
DEST.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(OUTPUT, DEST)

print("\nSUCCESS — sequence features generated")
print("Saved to:", OUTPUT)
print("Backed up to:", DEST)

In [ ]:
from pathlib import Path
import subprocess
import shutil

ROOT = Path("/content/NMDpredictionmodel")
BACKUP = Path("/content/drive/MyDrive/NMD_ClinVar_demo_checkpoints")

SCRIPT = ROOT / "Features/01b_GENCODEv26_intron_features.py"
GTF = ROOT / "Annotation/reference/clinvar20_gencode_v26.gtf"

SEQ_RDS = ROOT / "Features/output/ClinVar_gencode_v26_sequence_features.rds"

INPUT_CSV = ROOT / "Features/output/ClinVar_01b_input.csv"
OUTPUT_CSV = ROOT / "Features/output/ClinVar_01b_intron_features.csv"

for f in [SCRIPT, GTF, SEQ_RDS]:
    assert f.is_file(), f"Missing required file: {f}"

# 1. Convert Step 01a RDS into Step 01b-compatible CSV
r_code = f'''
x <- readRDS("{SEQ_RDS}")

stopifnot(
    is.data.frame(x),
    "txnames" %in% names(x),
    !anyDuplicated(x$txnames)
)

for (nm in c(
    "cdsseq.introns",
    "threeUTR.introns",
    "fiveUTR.introns"
)) {{
    if (!nm %in% names(x)) x[[nm]] <- NA_character_
}}

write.csv(
    x,
    file="{INPUT_CSV}",
    row.names=FALSE,
    na=""
)

cat("Input transcripts:", nrow(x), "\\n")
'''

p = subprocess.run(
    ["Rscript", "-e", r_code],
    capture_output=True,
    text=True
)

print(p.stdout)
if p.returncode:
    raise RuntimeError(p.stderr)

# 2. Run the original GitHub Step 01b script
p = subprocess.run(
    [
        "python", str(SCRIPT),
        "--table", str(INPUT_CSV),
        "--gtf", str(GTF),
        "--output", str(OUTPUT_CSV)
    ],
    capture_output=True,
    text=True
)

print(p.stdout)
if p.returncode:
    raise RuntimeError(p.stderr)

# 3. Validate output
import pandas as pd

df = pd.read_csv(OUTPUT_CSV)

expected = [
    "txnames",
    "cdsseq.introns",
    "threeUTR.introns",
    "fiveUTR.introns"
]

assert all(c in df.columns for c in expected)
assert len(df) == 12, f"Expected 12 transcripts, found {len(df)}"
assert df["txnames"].is_unique, "Duplicate transcript IDs"

print("\n=== INTRON FEATURE COUNTS ===")

for col in expected[1:]:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))

# 4. Back up to Google Drive
DEST = BACKUP / "Features/output" / OUTPUT_CSV.name
DEST.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(OUTPUT_CSV, DEST)

print("\nSTEP 01b COMPLETE")
print("Output:", OUTPUT_CSV)
print("Drive backup:", DEST)

In [ ]:
from pathlib import Path
import subprocess
import shutil

ROOT = Path("/content/NMDpredictionmodel")
BACKUP = Path("/content/drive/MyDrive/NMD_ClinVar_demo_checkpoints")

GTF = ROOT / "Annotation/reference/clinvar20_gencode_v26.gtf"
SCRIPT = Path("/content/ClinVar_02_PTBP1_corrected.R")
OUTPUT = ROOT / "Features/output/ClinVar_PTBP1_3UTR_features.rds"

BEDS = [
    ROOT / "Features/reference/ENCFF907HNN.bed",
    ROOT / "Features/reference/ENCFF130PWU.bed.txt",
    ROOT / "Features/reference/ENCFF100OEX.bed.txt",
]

for path in [GTF, *BEDS]:
    assert path.is_file(), f"Missing input: {path}"

SCRIPT.write_text(r'''
suppressPackageStartupMessages({
    library(GenomicFeatures)
    library(GenomicRanges)
    library(GenomeInfoDb)
    library(rtracklayer)
})

args <- commandArgs(trailingOnly = TRUE)
stopifnot(length(args) >= 5)

gtf_file <- args[1]
output_file <- args[2]
bed_files <- args[-c(1, 2)]

txdb <- txdbmaker::makeTxDbFromGFF(gtf_file)

keep_chr <- intersect(
    paste0("chr", 1:22),
    GenomeInfoDb::seqlevels(txdb)
)

txdb <- GenomeInfoDb::keepSeqlevels(
    txdb, keep_chr, pruning.mode = "coarse"
)

utr <- GenomicFeatures::threeUTRsByTranscript(
    txdb, use.names = TRUE
)

message("Transcripts with 3UTR: ", length(utr))

# Take the first 400 spliced nucleotides in transcript orientation.
proximal <- GenomicRanges::GRangesList(
    lapply(utr, function(gr) {
        if (!length(gr)) return(gr)

        is_minus <- as.character(strand(gr)[1]) == "-"
        gr <- gr[order(start(gr), decreasing = is_minus)]

        remaining <- 400L
        pieces <- GenomicRanges::GRanges()

        for (i in seq_along(gr)) {
            if (remaining <= 0L) break

            take <- min(width(gr[i]), remaining)
            piece <- resize(gr[i], width = take, fix = "start")

            pieces <- c(pieces, piece)
            remaining <- remaining - take
        }

        pieces
    })
)

names(proximal) <- names(utr)

# Verify the correct number of retained spliced nucleotides.
expected <- pmin(400L, vapply(utr, function(x) sum(width(x)), numeric(1)))
observed <- vapply(proximal, function(x) sum(width(x)), numeric(1))
stopifnot(identical(as.numeric(observed), as.numeric(expected)))

read_bed <- function(path) {
    bed <- read.table(
        path, sep = "\t", header = FALSE,
        stringsAsFactors = FALSE, comment.char = "",
        quote = "", fill = TRUE
    )

    if (ncol(bed) < 6L)
        stop("BED lacks strand column: ", path)

    valid <- bed[[1]] %in% paste0("chr", 1:22) &
             bed[[6]] %in% c("+", "-") &
             !is.na(bed[[2]]) &
             !is.na(bed[[3]]) &
             bed[[3]] > bed[[2]]

    bed <- bed[valid, , drop = FALSE]

    GRanges(
        seqnames = bed[[1]],
        ranges = IRanges(
            start = as.integer(bed[[2]]) + 1L,
            end = as.integer(bed[[3]])
        ),
        strand = bed[[6]]
    )
}

peaks <- do.call(c, lapply(bed_files, read_bed))
message("PTBP1 peaks loaded: ", length(peaks))

# Use strand-aware genomic overlaps.
pieces <- unlist(proximal, use.names = FALSE)
pieces$tx <- rep(names(proximal), lengths(proximal))

hits <- findOverlaps(pieces, peaks, ignore.strand = FALSE)
hit_tx <- unique(pieces$tx[queryHits(hits)])

# Include all transcript IDs present in the TxDb.
all_tx <- names(GenomicFeatures::transcripts(txdb, columns = "tx_name"))
all_tx <- unique(all_tx[!is.na(all_tx)])

result <- data.frame(
    txnames = all_tx,
    threeUTR.PTBP1 = all_tx %in% hit_tx,
    stringsAsFactors = FALSE
)

stopifnot(!anyDuplicated(result$txnames))

saveRDS(result, output_file)

write.table(
    result,
    sub("\\.rds$", ".tsv", output_file),
    sep = "\t", quote = FALSE, row.names = FALSE
)

print(table(result$threeUTR.PTBP1))
message("PTBP1 feature extraction complete.")
''')

OUTPUT.parent.mkdir(parents=True, exist_ok=True)

cmd = [
    "Rscript", str(SCRIPT),
    str(GTF), str(OUTPUT),
    *map(str, BEDS)
]

print("Running PTBP1 feature extraction...")

p = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(p.stdout[-5000:])
print(p.stderr[-5000:])

if p.returncode != 0:
    raise RuntimeError("PTBP1 feature extraction failed")

assert OUTPUT.is_file(), "Output RDS not created"

# Save completed output to Google Drive
dest = BACKUP / "Features/output" / OUTPUT.name
dest.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(OUTPUT, dest)

tsv = OUTPUT.with_suffix(".tsv")
if tsv.exists():
    shutil.copy2(tsv, dest.with_suffix(".tsv"))

print("\nSTEP 02 COMPLETE")
print("Output:", OUTPUT)
print("Drive backup:", dest)

In [ ]:
from pathlib import Path
import subprocess
import shutil

ROOT = Path("/content/NMDpredictionmodel")
BACKUP = Path("/content/drive/MyDrive/NMD_ClinVar_demo_checkpoints")

GTF = ROOT / "Annotation/reference/clinvar20_gencode_v26.gtf"
INPUT = ROOT / "Annotation/output/ClinVar_PTC_filtered.rds"
OUTPUT = ROOT / "Features/output/ClinVar_PTC_amino_acid_features.rds"
SCRIPT = Path("/content/ClinVar_03_PTC_amino_acid_corrected.R")

for path in [GTF, INPUT]:
    assert path.is_file(), f"Missing input: {path}"

OUTPUT.parent.mkdir(parents=True, exist_ok=True)

SCRIPT.write_text(r'''
suppressPackageStartupMessages({
    library(GenomicFeatures)
    library(GenomicRanges)
    library(BSgenome.Hsapiens.UCSC.hg38)
    library(Biostrings)
})

args <- commandArgs(trailingOnly = TRUE)
stopifnot(length(args) == 3L)

gtf_file <- args[1]
input_file <- args[2]
output_file <- args[3]

variants <- readRDS(input_file)

required <- c(
    "key", "variantID", "txnames", "coding.pos",
    "ref", "alt", "V3"
)
stopifnot(all(required %in% names(variants)))

txdb <- txdbmaker::makeTxDbFromGFF(gtf_file)

keep_chr <- intersect(
    paste0("chr", 1:22),
    GenomeInfoDb::seqlevels(txdb)
)

txdb <- GenomeInfoDb::keepSeqlevels(
    txdb, keep_chr, pruning.mode = "coarse"
)

genome <- BSgenome.Hsapiens.UCSC.hg38

cds_by_tx <- GenomicFeatures::cdsBy(
    txdb, by = "tx", use.names = TRUE
)

cds_seqs <- GenomicFeatures::extractTranscriptSeqs(
    genome, cds_by_tx
)

# ANNOVAR's transcript versions may differ from the GTF.
# Match exact IDs first, then match versionless IDs only if unique.
match_tx <- function(id, available) {
    i <- match(id, available)
    if (!is.na(i)) return(i)

    base_id <- sub("\\.[0-9]+$", "", id)
    candidates <- which(
        sub("\\.[0-9]+$", "", available) == base_id
    )

    if (length(candidates) == 1L) return(candidates)
    NA_integer_
}

# Parse the coding substitution belonging to the selected transcript.
parse_coding_change <- function(annotation, tx) {
    entries <- strsplit(annotation, ",", fixed = TRUE)[[1]]
    selected <- entries[
        vapply(
            entries,
            function(z) {
                fields <- strsplit(z, ":", fixed = TRUE)[[1]]
                length(fields) >= 5L && fields[2] == tx
            },
            logical(1)
        )
    ]

    if (length(selected) != 1L)
        return(c(NA_character_, NA_character_))

    fields <- strsplit(selected, ":", fixed = TRUE)[[1]]
    coding <- fields[4]

    m <- regexec(
        "^c\\.([0-9]+)([ACGT])>([ACGT])$|^c\\.([ACGT])([0-9]+)([ACGT])$",
        coding
    )

    # Standard ANNOVAR format is c.C2107T (without >).
    m2 <- regexec("^c\\.([ACGT])([0-9]+)([ACGT])$", coding)
    hit <- regmatches(coding, m2)[[1]]

    if (length(hit) != 4L)
        return(c(NA_character_, NA_character_))

    c(hit[2], hit[4])
}

aa_at <- function(protein, position) {
    if (
        is.na(position) ||
        position < 1L ||
        position > nchar(protein)
    ) return(NA_character_)

    substr(protein, position, position)
}

n <- nrow(variants)

protein_position <- rep(NA_integer_, n)
minus2 <- minus1 <- plus1 <- plus2 <- rep(NA_character_, n)
ptc_id <- rep(NA_character_, n)
qc_status <- rep("UNPROCESSED", n)

for (i in seq_len(n)) {
    tx <- as.character(variants$txnames[i])
    pos <- as.integer(variants$coding.pos[i])

    j <- match_tx(tx, names(cds_seqs))

    if (is.na(j)) {
        qc_status[i] <- "TRANSCRIPT_NOT_FOUND"
        next
    }

    cds <- as.character(cds_seqs[[j]])

    if (is.na(pos) || pos < 1L || pos > nchar(cds)) {
        qc_status[i] <- "CODING_POSITION_OUT_OF_RANGE"
        next
    }

    coding_change <- parse_coding_change(
        as.character(variants$V3[i]), tx
    )

    ref_coding <- coding_change[1]
    alt_coding <- coding_change[2]

    if (anyNA(coding_change)) {
        qc_status[i] <- "ANNOTATION_PARSE_FAILED"
        next
    }

    observed_ref <- substr(cds, pos, pos)

    if (observed_ref != ref_coding) {
        qc_status[i] <- "REFERENCE_MISMATCH"
        next
    }

    codon_start <- ((pos - 1L) %/% 3L) * 3L + 1L
    reference_codon <- substr(cds, codon_start, codon_start + 2L)

    if (nchar(reference_codon) != 3L) {
        qc_status[i] <- "INCOMPLETE_CODON"
        next
    }

    mutant_codon <- reference_codon
    changed_base <- pos - codon_start + 1L

    substr(
        mutant_codon, changed_base, changed_base
    ) <- alt_coding

    if (!mutant_codon %in% c("TAA", "TAG", "TGA")) {
        qc_status[i] <- "MUTANT_NOT_STOP"
        next
    }

    aa_position <- (pos - 1L) %/% 3L + 1L
    protein_position[i] <- aa_position

    # Translate the reference CDS; the variant creates the stop.
    reference_protein <- as.character(
        Biostrings::translate(
            Biostrings::DNAString(cds),
            if.fuzzy.codon = "X"
        )
    )

    minus2[i] <- aa_at(reference_protein, aa_position - 2L)
    minus1[i] <- aa_at(reference_protein, aa_position - 1L)
    plus1[i] <- aa_at(reference_protein, aa_position + 1L)
    plus2[i] <- aa_at(reference_protein, aa_position + 2L)

    ptc_id[i] <- mutant_codon
    qc_status[i] <- "PASS"
}

result <- data.frame(
    key = variants$key,
    variantID = variants$variantID,
    txnames = variants$txnames,
    protein_position = protein_position,
    PTC.minus2.aa = minus2,
    PTC.minus1.aa = minus1,
    PTC.plus1.aa = plus1,
    PTC.plus2.aa = plus2,
    PTC.id = ptc_id,
    QC_status = qc_status,
    stringsAsFactors = FALSE
)

cat("\nQC status:\n")
print(table(result$QC_status))

cat("\nStop codons:\n")
print(table(result$PTC.id, useNA = "ifany"))

stopifnot(
    nrow(result) == nrow(variants),
    all(result$QC_status == "PASS"),
    all(result$PTC.id %in% c("TAA", "TAG", "TGA"))
)

saveRDS(result, output_file)
message("PTC amino-acid context extraction complete.")
''')

print("Running Step 03...")

p = subprocess.run(
    ["Rscript", str(SCRIPT), str(GTF), str(INPUT), str(OUTPUT)],
    capture_output=True,
    text=True
)

print(p.stdout[-6000:])
print(p.stderr[-6000:])

if p.returncode != 0:
    raise RuntimeError("Step 03 failed; output was not backed up")

assert OUTPUT.is_file(), "Output RDS missing"

dest = BACKUP / "Features/output" / OUTPUT.name
dest.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(OUTPUT, dest)

print("\nSTEP 03 COMPLETE — QC PASSED")
print("Output:", OUTPUT)
print("Drive backup:", dest)

In [ ]:
from pathlib import Path
import subprocess
import shutil

ROOT = Path("/content/NMDpredictionmodel")
SCRIPT = Path("/content/ClinVar_04_gene_level_corrected.R")
BACKUP = Path("/content/drive/MyDrive/NMD_ClinVar_demo_checkpoints")

source = SCRIPT.read_text()

# -----------------------------------------------------------
# Fix Section 2: join on stable transcript IDs
# -----------------------------------------------------------

old = '''variants <- variants %>%
    left_join(
        gene_map,
        by = c("txnames" = "ensembl_transcript_id"),
        relationship = "many-to-one"
    )'''

new = '''# Match transcript IDs without version suffixes.
# This mapping is used only to assign stable gene IDs.

gene_map <- gene_map %>%
    mutate(
        transcript_base = sub("\\\\..*$", "", ensembl_transcript_id),
        ensembl_gene_id = sub("\\\\..*$", "", ensembl_gene_id)
    )

stopifnot(!anyDuplicated(gene_map$transcript_base))

variants <- variants %>%
    mutate(
        transcript_base = sub("\\\\..*$", "", txnames)
    ) %>%
    left_join(
        gene_map %>%
            select(transcript_base, ensembl_gene_id, hgnc_symbol),
        by = "transcript_base",
        relationship = "many-to-one"
    ) %>%
    select(-transcript_base)'''

assert old in source, "Original mapping block not found; no changes made."

source = source.replace(old, new, 1)

# -----------------------------------------------------------
# Strengthen final QC
# -----------------------------------------------------------

old_qc = '''saveRDS(variants, CONFIG$output_file)'''

new_qc = '''# Require successful transcript-to-gene mapping.
stopifnot(
    sum(!is.na(variants$ensembl_gene_id)) == original_n
)

# Ensure gene-level annotation is not entirely missing.
stopifnot(
    sum(!is.na(variants$pLI)) > 0,
    sum(!is.na(variants$oe_lof_upper)) > 0,
    sum(!is.na(variants$MedianExpression)) > 0,
    sum(!is.na(variants$Whole.Blood)) > 0,
    sum(!is.na(variants$half_life_PC1)) > 0
)

saveRDS(variants, CONFIG$output_file)'''

assert old_qc in source, "Save block not found; no changes made."

source = source.replace(old_qc, new_qc, 1)
SCRIPT.write_text(source)

# -----------------------------------------------------------
# Run corrected Step 04
# -----------------------------------------------------------

GTF = ROOT / "Annotation/reference/clinvar20_gencode_v26.gtf"
INPUT = ROOT / "Annotation/output/ClinVar_PTC_filtered.rds"

GNOMAD = Path(
    "/content/drive/MyDrive/gnomad.v4.1.constraint_metrics.tsv"
)

GTEX = ROOT / (
    "Features/reference/"
    "GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_median_tpm.gct.txt"
)

HALF_LIFE = ROOT / "Model/TrunKitten/inputs/half_life_pc1.xlsx"

OUTPUT = ROOT / "Features/output/ClinVar_gene_level_features.rds"

result = subprocess.run(
    [
        "Rscript",
        str(SCRIPT),
        str(GTF),
        str(INPUT),
        str(GNOMAD),
        str(GTEX),
        str(HALF_LIFE),
        str(OUTPUT),
    ],
    capture_output=True,
    text=True
)

print(result.stdout[-6000:])
print(result.stderr[-3000:])

if result.returncode != 0:
    raise RuntimeError(
        "Step 04 failed QC. Existing Drive backup was not overwritten."
    )

assert OUTPUT.is_file()

destination = BACKUP / "Features/output" / OUTPUT.name
destination.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(OUTPUT, destination)

print("\nSTEP 04 COMPLETE — QC PASSED")
print("Output:", OUTPUT)
print("Drive backup:", destination)

In [ ]:
from pathlib import Path
import subprocess
import shutil

ROOT = Path("/content/NMDpredictionmodel")
BACKUP = Path("/content/drive/MyDrive/NMD_ClinVar_demo_checkpoints")

ORIGINAL = ROOT / "Features/05_PTC_geometry_derived_features.R"
INPUT = ROOT / "Annotation/output/ClinVar_PTC_filtered.rds"
OUTPUT = ROOT / "Features/output/ClinVar_PTC_geometry_features.rds"
SCRIPT = Path("/content/ClinVar_05_PTC_geometry_corrected.R")

assert ORIGINAL.is_file()
assert INPUT.is_file()

# Preserve the original GitHub formulas and feature definitions.
source = ORIGINAL.read_text()

# Replace only the input/output configuration paths.
source = source.replace(
    '"/path/to/PTC_annotated_variants.rds"',
    f'"{INPUT}"'
).replace(
    '"/path/to/PTC_geometry_features.rds"',
    f'"{OUTPUT}"'
)

assert str(INPUT) in source
assert str(OUTPUT) in source

# Add validation before the original calculations.
marker = "# ------------------------------------------------------------------------------\n# 5. Transcript-relative PTC geometry"

qc = r'''
# Additional QC: verify CDS exon indexing and coordinates.
cds_ends <- strsplit(as.character(variants$cds_exons), ",")

for (i in seq_len(nrow(variants))) {
    ends <- suppressWarnings(as.numeric(cds_ends[[i]]))
    exon_idx <- as.integer(variants$mut.exon[i])
    pos <- as.numeric(variants$coding.pos[i])
    cds_len <- as.numeric(variants$cds_length[i])
    exon_len <- as.numeric(variants$length.mutated.exon[i])

    stopifnot(
        length(ends) == as.integer(variants$exon_count[i]),
        all(is.finite(ends)),
        all(diff(ends) > 0),
        tail(ends, 1) == cds_len,
        exon_idx >= 1,
        exon_idx <= length(ends),
        pos >= 1,
        pos <= cds_len
    )

    exon_start <- if (exon_idx == 1) 1 else ends[exon_idx - 1] + 1
    exon_end <- ends[exon_idx]

    stopifnot(
        pos >= exon_start,
        pos <= exon_end,
        exon_len == exon_end - exon_start + 1
    )
}

cat("CDS exon indexing and coordinate QC: PASSED\n\n")

'''

assert marker in source, "Could not locate Step 05 calculation section."
source = source.replace(marker, qc + marker, 1)

# Add final validation immediately before saving.
save_marker = '''saveRDS(
    PTC_geometry_features,
    CONFIG$output_file
)'''

final_qc = r'''
stopifnot(
    nrow(PTC_geometry_features) == 13L,
    identical(
        as.character(PTC_geometry_features$key),
        as.character(readRDS(CONFIG$input_file)$key)
    ),
    all(!is.na(PTC_geometry_features$relativePTClocation)),
    all(!is.na(PTC_geometry_features$relPTC_exon)),
    all(PTC_geometry_features$relativePTClocation >= 0 &
        PTC_geometry_features$relativePTClocation <= 1),
    all(PTC_geometry_features$relPTC_exon >= 0 &
        PTC_geometry_features$relPTC_exon <= 1),
    all(PTC_geometry_features$AmountExonsAfter >= 0)
)

cat("STEP 05 FINAL QC: PASSED\n")
'''

assert save_marker in source, "Could not locate original save statement."
source = source.replace(save_marker, final_qc + "\n" + save_marker, 1)

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
SCRIPT.write_text(source)

result = subprocess.run(
    ["Rscript", str(SCRIPT)],
    capture_output=True,
    text=True
)

print(result.stdout[-5000:])
print(result.stderr[-5000:])

if result.returncode != 0:
    raise RuntimeError(
        "Step 05 failed QC. No new Drive backup was created."
    )

assert OUTPUT.is_file()

destination = BACKUP / "Features/output" / OUTPUT.name
destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(OUTPUT, destination)

print("\nSTEP 05 COMPLETE")
print("Output:", OUTPUT)
print("Drive backup:", destination)

In [ ]:
from pathlib import Path
import pandas as pd
import subprocess
import shutil

ROOT = Path("/content/NMDpredictionmodel")
BACKUP = Path("/content/drive/MyDrive/NMD_ClinVar_demo_checkpoints")

AVINPUT = ROOT / "examples/ClinVar_variants_for_external_annotation.avinput"
SCORES = ROOT / "examples/ClinVar_variants_external_annotation.hg38_multianno.txt"
VARIANTS = ROOT / "Annotation/output/ClinVar_PTC_filtered.rds"
OUTPUT = ROOT / "Features/output/ClinVar_variant_annotation_scores.rds"

for path in [AVINPUT, SCORES, VARIANTS]:
    assert path.is_file(), f"Missing required file: {path}"

# ----------------------------------------------------------
# 1. Validate Step 06a and Step 06b
# ----------------------------------------------------------

input_df = pd.read_csv(
    AVINPUT,
    sep="\t",
    header=None,
    names=["Chr", "Start", "End", "Ref", "Alt"]
)

score_df = pd.read_csv(
    SCORES,
    sep="\t",
    na_values=[".", "NA", ""]
)

required = [
    "Chr", "Start", "Ref", "Alt",
    "CADD_raw", "CADD_phred", "gnomAD_exome_ALL"
]

assert all(c in score_df.columns for c in required)

def make_key(df):
    return (
        "chr" + df["Chr"].astype(str).str.replace(
            r"^chr", "", regex=True
        )
        + ":" + df["Start"].astype(str)
        + "_" + df["Ref"].astype(str)
        + ">" + df["Alt"].astype(str)
    )

input_keys = make_key(input_df)
score_keys = make_key(score_df)

assert len(input_df) == 13
assert len(score_df) == 13
assert set(input_keys) == set(score_keys)
assert score_keys.nunique() == 12

print("STEP 06a–06b QC PASSED")
print("Input records:", len(input_df))
print("Annotation records:", len(score_df))
print("Unique genomic variants:", score_keys.nunique())

# ----------------------------------------------------------
# 2. Run Step 06c in R
# ----------------------------------------------------------

SCRIPT = Path("/content/ClinVar_06c_merge_variant_scores_corrected.R")

SCRIPT.write_text(r'''
library(dplyr)

CONFIG <- list(
    variant_file =
        "/content/NMDpredictionmodel/Annotation/output/ClinVar_PTC_filtered.rds",

    score_file =
        "/content/NMDpredictionmodel/examples/ClinVar_variants_external_annotation.hg38_multianno.txt",

    output_file =
        "/content/NMDpredictionmodel/Features/output/ClinVar_variant_annotation_scores.rds"
)

variants <- readRDS(CONFIG$variant_file)

scores <- read.delim(
    CONFIG$score_file,
    stringsAsFactors = FALSE,
    check.names = FALSE,
    na.strings = c(".", "NA", "")
)

scores$key <- with(
    scores,
    paste0(
        ifelse(grepl("^chr", Chr), "", "chr"),
        Chr, ":", Start, "_", Ref, ">", Alt
    )
)

scores_to_merge <- scores %>%
    select(
        key,
        CADD_raw,
        CADD_phred,
        gnomAD_exome_ALL
    )

# Ensure repeated genomic variants have identical annotations
conflicts <- scores_to_merge %>%
    distinct() %>%
    count(key) %>%
    filter(n > 1)

if (nrow(conflicts) > 0) {
    stop(
        "Conflicting annotations for: ",
        paste(conflicts$key, collapse = ", ")
    )
}

scores_to_merge <- scores_to_merge %>%
    distinct(key, .keep_all = TRUE)

variants_with_scores <- variants %>%
    left_join(
        scores_to_merge,
        by = "key",
        relationship = "many-to-one"
    )

stopifnot(
    nrow(variants_with_scores) == nrow(variants),
    nrow(variants_with_scores) == 13L,
    identical(variants_with_scores$key, variants$key),
    all(variants$key %in% scores_to_merge$key)
)

for (col in c("CADD_raw", "CADD_phred", "gnomAD_exome_ALL")) {
    variants_with_scores[[col]] <- as.numeric(
        variants_with_scores[[col]]
    )
}

variant_annotation_scores <- variants_with_scores %>%
    select(
        key,
        CADD_raw,
        CADD_phred,
        gnomAD_exome_ALL
    )

saveRDS(
    variant_annotation_scores,
    CONFIG$output_file
)

cat("STEP 06c QC PASSED\n")
cat("Input variants:", nrow(variants), "\n")
cat("Output variants:", nrow(variant_annotation_scores), "\n")

for (col in c("CADD_raw", "CADD_phred", "gnomAD_exome_ALL")) {
    cat(
        col, ":",
        sum(!is.na(variant_annotation_scores[[col]])),
        "/ 13 non-missing\n"
    )
}

print(variant_annotation_scores)
''')

OUTPUT.parent.mkdir(parents=True, exist_ok=True)

result = subprocess.run(
    ["Rscript", str(SCRIPT)],
    capture_output=True,
    text=True
)

print("\n" + result.stdout)
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Step 06c failed. No backup created.")

assert OUTPUT.is_file()

destination = BACKUP / "Features/output" / OUTPUT.name
destination.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(OUTPUT, destination)

print("\nSTEP 06 COMPLETE — QC PASSED")
print("Output:", OUTPUT)
print("Drive backup:", destination)

## 6. Motif scanning and sequence context

In [ ]:
from pathlib import Path
import requests
import time
import gffutils

ROOT = Path("/content/NMDpredictionmodel")
DB_PATH = ROOT / "Annotation/reference/clinvar20_gencode_v26.db"
REF_DIR = ROOT / "Annotation/reference"
REF_DIR.mkdir(parents=True, exist_ok=True)

db = gffutils.FeatureDB(str(DB_PATH))

# Download each transcript interval independently.
# Ensembl region endpoint returns forward genomic sequence.
regions = {}

for tx in db.features_of_type("transcript"):
    chrom = tx.chrom.replace("chr", "")
    regions[tx.id] = {
        "chrom": tx.chrom,
        "start": tx.start,
        "end": tx.end
    }

session = requests.Session()
session.headers.update({"Accept": "text/plain"})

seq_lookup = {}

for txid, info in regions.items():
    chrom = info["chrom"].replace("chr", "")
    start, end = info["start"], info["end"]

    url = (
        "https://rest.ensembl.org/sequence/region/human/"
        f"{chrom}:{start}..{end}:1"
    )

    for attempt in range(5):
        try:
            response = session.get(
                url,
                params={"coord_system_version": "GRCh38"},
                timeout=90
            )
            response.raise_for_status()
            seq = response.text.strip().upper()

            assert len(seq) == end - start + 1
            seq_lookup[txid] = seq
            break
        except Exception:
            if attempt == 4:
                raise
            time.sleep(3 * (attempt + 1))

    print(txid, len(seq_lookup[txid]), "bp")

print("\nDownloaded transcripts:", len(seq_lookup))

In [ ]:
# Preserve the original function
original_build_regions = motif.build_regions

# Replace the function's global database access with a
# version-independent lookup adapter.
class VersionFreeDB:
    def __init__(self, db):
        self.db = db

    def __getitem__(self, txid):
        stable_id = str(txid).split(".")[0]
        matched_id = tx_lookup.get(stable_id)

        if matched_id is None:
            raise KeyError(f"Transcript not found: {txid}")

        return self.db[matched_id]

    def children(self, *args, **kwargs):
        return self.db.children(*args, **kwargs)

version_free_db = VersionFreeDB(db)

# Make the existing four-argument call compatible
def build_regions_fixed(genome, db, row, tx_lookup):
    return original_build_regions(
        genome,
        version_free_db,
        row
    )

motif.build_regions = build_regions_fixed

print("Patched:", inspect.signature(motif.build_regions))

In [ ]:
from pathlib import Path
import pandas as pd
import gffutils
import importlib.util

ROOT = Path("/content/NMDpredictionmodel")
SCRIPT = ROOT / "Features/07a_motif_region_extraction.py"
INPUT = Path("/content/ClinVar_step07a_input_check.csv")
DB_PATH = ROOT / "Annotation/reference/clinvar20_gencode_v26.db"
OUTPUT = ROOT / "Features/output/ClinVar_variants_with_motif_regions.csv"

OUTPUT.parent.mkdir(parents=True, exist_ok=True)

# Reuse downloaded sequences from Cell 1
assert "seq_lookup" in globals(), "Cell 1 sequences missing."
assert "regions" in globals(), "Cell 1 region coordinates missing."

# Load original script
spec = importlib.util.spec_from_file_location("motif07a", SCRIPT)
motif = importlib.util.module_from_spec(spec)
spec.loader.exec_module(motif)

df = pd.read_csv(INPUT)
db = gffutils.FeatureDB(str(DB_PATH))

tx_lookup = {
    tx.id.split(".")[0]: tx.id
    for tx in db.features_of_type("transcript")
}

# Allow original three-argument build_regions() to find
# transcripts regardless of their version number.
class VersionFreeDB:
    def __init__(self, db):
        self.db = db

    def __getitem__(self, txid):
        stable_id = str(txid).split(".")[0]
        matched_id = tx_lookup.get(stable_id)
        if matched_id is None:
            raise KeyError(f"Transcript not found: {txid}")
        return self.db[matched_id]

    def children(self, *args, **kwargs):
        return self.db.children(*args, **kwargs)

version_free_db = VersionFreeDB(db)

class SequenceFragment:
    def __init__(self, seq):
        self.seq = seq

class ChromosomeAccessor:
    def __init__(self, info, seq):
        self.info = info
        self.seq = seq

    def __getitem__(self, region):
        start0 = region.start
        end0 = region.stop

        offset = self.info["start"] - 1
        local_start = start0 - offset
        local_end = end0 - offset

        if local_start < 0 or local_end > len(self.seq):
            raise ValueError("Sequence request outside transcript interval")

        return SequenceFragment(self.seq[local_start:local_end])

class TranscriptGenome:
    def __init__(self, info, seq):
        self.info = info
        self.seq = seq

    def __getitem__(self, chrom):
        if chrom != self.info["chrom"]:
            raise KeyError(chrom)
        return ChromosomeAccessor(self.info, self.seq)

results = []

for index, row in df.iterrows():
    stable_id = str(row["txnames"]).split(".")[0]
    txid = tx_lookup[stable_id]

    genome = TranscriptGenome(regions[txid], seq_lookup[txid])

    adjusted_row = row.copy()
    adjusted_row["cds_length"] = int(row["cds_length"]) - 3

    # Compatible with both script versions
    import inspect
    n_args = len(inspect.signature(motif.build_regions).parameters)

    if n_args == 4:
        region_sequences = motif.build_regions(
            genome, db, adjusted_row, tx_lookup
        )
    else:
        region_sequences = motif.build_regions(
            genome, version_free_db, adjusted_row
        )

    results.append(region_sequences)
    print(f"Processed {index + 1}/{len(df)}: {row['key']}")

regions_df = pd.DataFrame(results)

final_df = pd.concat(
    [df.reset_index(drop=True), regions_df.reset_index(drop=True)],
    axis=1
)

final_df.to_csv(OUTPUT, index=False)

print("\nSaved:", OUTPUT)

print("\n=== REGION AVAILABILITY ===")
for region in motif.REGIONS:
    if region in final_df.columns:
        count = final_df[region].fillna("").ne("").sum()
        print(f"{region}: {count}/{len(final_df)}")
    else:
        print(f"{region}: MISSING")

In [ ]:
from pathlib import Path
import pandas as pd
import shutil

ROOT = Path("/content/NMDpredictionmodel")
INPUT = ROOT / "Features/output/ClinVar_variants_with_motif_regions.csv"
OUT_DIR = ROOT / "Features/output/ClinVar_FIMO_input"
BACKUP_DIR = Path(
    "/content/drive/MyDrive/NMD_ClinVar_demo_checkpoints/"
    "Features/output/ClinVar_FIMO_input"
)

REGIONS = [
    "ptc_to_ejc",
    "ptc_pm100",
    "ejc_pm100",
    "newutr_all",
    "newutr_200",
    "utr3_all",
    "utr3_200"
]

df = pd.read_csv(INPUT)

assert len(df) == 13
assert all(r in df.columns for r in REGIONS)

OUT_DIR.mkdir(parents=True, exist_ok=True)

# Unique, stable row identifiers
df.insert(
    0,
    "fasta_id",
    [f"ClinVar_{i:05d}" for i in range(1, len(df) + 1)]
)

# Save mapping for joining FIMO results later
mapping = df[["fasta_id", "key", "txnames"]].copy()
mapping.to_csv(OUT_DIR / "ClinVar_FASTA_ID_mapping.csv", index=False)

# Create one FASTA per motif region
for region in REGIONS:
    fasta_path = OUT_DIR / f"{region}.fasta"

    with open(fasta_path, "w") as handle:
        for _, row in df.iterrows():
            seq = str(row[region]).upper()

            if not seq or seq == "NAN":
                continue

            handle.write(f">{row['fasta_id']}\n{seq}\n")

    print(f"{region}: {fasta_path}")

# Backup
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

for path in OUT_DIR.iterdir():
    if path.is_file():
        shutil.copy2(path, BACKUP_DIR / path.name)

print("\n=== QC ===")
print("Variants:", len(df))
print("Unique FASTA IDs:", df["fasta_id"].nunique())
print("FASTA files:", len(REGIONS))
print("Mapping rows:", len(mapping))
print("\nGoogle Drive backup:", BACKUP_DIR)

In [ ]:
from pathlib import Path
import importlib.util
import pandas as pd
import os

ROOT = Path("/content/NMDpredictionmodel")
SCRIPT = ROOT / "Features/07b_FIMO_motif_features.py"

spec = importlib.util.spec_from_file_location("motif07b", SCRIPT)
motif = importlib.util.module_from_spec(spec)
spec.loader.exec_module(motif)

# Configure original script for this ClinVar demo
motif.WORKDIR = ROOT / "Features"
motif.SEQUENCE_CSV = ROOT / "Features/output/ClinVar_variants_with_motif_regions.csv"
motif.FASTA_DIR = ROOT / "Features/output/ClinVar_FIMO_input"
motif.MOTIFS = ROOT / "Features/reference/memefile.txt"
motif.ATTRACT = ROOT / "Features/reference/ATtRACT_db_human.txt"
motif.OUTPUT_DIR = ROOT / "Features/output/ClinVar_FIMO_results"

motif.N_CORES = 2

# Make installed FIMO available to the original script
os.environ["PATH"] = "/content/meme_env/bin:" + os.environ["PATH"]

# Read actual ATtRACT gene names
attract = pd.read_csv(motif.ATTRACT, sep="\t")
valid_genes = set(attract["Gene_name"].dropna().astype(str).str.strip())

# Fix motif-to-RBP mapping for IDs such as 1000:PTBP1:ATtRACT
def map_motif_to_gene_fixed(motif_id, id_to_gene):
    parts = str(motif_id).split(":")

    if len(parts) >= 3 and parts[-1] == "ATtRACT":
        gene = parts[1].strip()
        return gene if gene in valid_genes else ""

    return motif.map_motif_to_gene_original(motif_id, id_to_gene)

motif.map_motif_to_gene_original = motif.map_motif_to_gene
motif.map_motif_to_gene = map_motif_to_gene_fixed

# Validate mapping before running FIMO
meme_ids = [
    line.split()[1]
    for line in motif.MOTIFS.read_text().splitlines()
    if line.startswith("MOTIF ")
]

mapped = [
    motif.map_motif_to_gene(m, {})
    for m in meme_ids
]

print("=== MOTIF MAPPING QC ===")
print("MEME motifs:", len(meme_ids))
print("Mapped to ATtRACT genes:", sum(bool(g) for g in mapped))
print("Unmapped:", sum(not g for g in mapped))

print("\nExamples:")
for motif_id, gene in list(zip(meme_ids, mapped))[:10]:
    print(motif_id, "->", gene)

print("\nFIMO version:")
motif.check_fimo()

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/NMDpredictionmodel")

FASTA_DIR = ROOT / "Features/output/ClinVar_FIMO_input"
OUTPUT_DIR = ROOT / "Features/output/ClinVar_FIMO_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

REGIONS = [
    "utr3_all",
    "utr3_200",
    "ptc_to_ejc",
    "newutr_all",
    "newutr_200",
    "ptc_pm100",
    "ejc_pm100"
]

all_results = []

print("=== RUNNING FIMO ===")

for region in REGIONS:
    fasta_path = FASTA_DIR / f"{region}.fasta"
    outdir = OUTPUT_DIR / "fimo_hits" / region

    assert fasta_path.exists(), f"Missing FASTA: {fasta_path}"

    print(f"\nScanning: {region}", flush=True)

    # Use the original manuscript function
    hits = motif.run_fimo_batch(
        (region, fasta_path, outdir)
    )

    if hits.empty:
        print(f"{region}: 0 hits")
    else:
        print(f"{region}: {len(hits):,} hits")
        all_results.append(hits)

# Combine all regions
if all_results:
    all_hits = pd.concat(all_results, ignore_index=True)
else:
    all_hits = pd.DataFrame()

HITS_FILE = OUTPUT_DIR / "all_fimo_hits.csv"
all_hits.to_csv(HITS_FILE, index=False)

print("\n=== FIMO SUMMARY ===")
print("Regions scanned:", len(REGIONS))
print("Total significant motif hits:", len(all_hits))
print("Saved:", HITS_FILE)

if not all_hits.empty:
    print("Unique FASTA IDs with hits:",
          all_hits["sequence_name"].nunique())
    print("Unique motif IDs detected:",
          all_hits["motif_id"].nunique())

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import shutil

ROOT = Path("/content/NMDpredictionmodel")
OUT = ROOT / "Features/output/ClinVar_FIMO_results"
FASTA_DIR = ROOT / "Features/output/ClinVar_FIMO_input"

hits = pd.read_csv(OUT / "all_fimo_hits.csv")
mapping = pd.read_csv(FASTA_DIR / "ClinVar_FASTA_ID_mapping.csv")

regions = [
    "utr3_all", "utr3_200", "ptc_to_ejc",
    "newutr_all", "newutr_200",
    "ptc_pm100", "ejc_pm100"
]

assert len(mapping) == 13
assert mapping["fasta_id"].is_unique
assert {"sequence_name", "motif_id", "region"}.issubset(hits.columns)

# Use the verified ATtRACT mapping
hits["gene"] = hits["motif_id"].apply(
    lambda x: motif.map_motif_to_gene(x, {})
)

assert hits["gene"].ne("").all(), "Unmapped motif IDs found"
assert hits["sequence_name"].isin(mapping["fasta_id"]).all()

# Feature naming from original 07b script
hits["feature"] = hits["region"] + "." + hits["gene"]

# Original logic: represent each detected RBP across all seven regions
genes = sorted(hits["gene"].unique())
features = [f"{r}.{g}" for r in regions for g in genes]

matrix = pd.DataFrame(
    0,
    index=mapping["fasta_id"],
    columns=features,
    dtype=np.int8
)
matrix.index.name = "fasta_id"

# Set motif presence to 1
presence = hits[["sequence_name", "feature"]].drop_duplicates()

for fasta_id, feature in presence.itertuples(index=False, name=None):
    matrix.at[fasta_id, feature] = 1

matrix = matrix.reset_index()

# Preserve original variant and transcript identifiers
matrix = mapping.merge(
    matrix,
    on="fasta_id",
    how="left",
    validate="one_to_one"
)

OUTPUT = OUT / "motif_presence_matrix.csv"
matrix.to_csv(OUTPUT, index=False)

# QC
feature_values = matrix[features].to_numpy()

assert len(matrix) == 13
assert matrix["fasta_id"].is_unique
assert set(np.unique(feature_values)).issubset({0, 1})
assert int(feature_values.sum()) == len(presence)

print("=== STEP 07b MATRIX QC ===")
print("Records:", len(matrix))
print("Unique variant keys:", matrix["key"].nunique())
print("RBP genes represented:", len(genes))
print("Binary motif feature columns:", len(features))
print("Positive feature entries:", int(feature_values.sum()))
print("Unmapped motif hits:", hits["gene"].eq("").sum())
print("Saved:", OUTPUT)

# Back up completed Step 07b results
BACKUP = Path(
    "/content/drive/MyDrive/NMD_ClinVar_demo_checkpoints/"
    "Features/output/ClinVar_FIMO_results"
)
BACKUP.mkdir(parents=True, exist_ok=True)

shutil.copy2(OUTPUT, BACKUP / OUTPUT.name)
shutil.copy2(
    OUT / "all_fimo_hits.csv",
    BACKUP / "all_fimo_hits.csv"
)

print("Google Drive backup:", BACKUP)

In [ ]:
from pathlib import Path
import pandas as pd
import runpy
import sys
import shutil

ROOT = Path("/content/NMDpredictionmodel")
OUT = ROOT / "Features/output"
OUT.mkdir(parents=True, exist_ok=True)

SOURCE = OUT / "ClinVar_variants_with_motif_regions.csv"
INPUT = OUT / "ClinVar_step08b_input.csv"
RESULT = OUT / "ClinVar_codon_optimality_features.tsv"

SCRIPT = ROOT / "Features/08b_codon_optimality_features.py"
GTF = ROOT / "Annotation/reference/clinvar20_gencode_v26.gtf"
FASTA = Path("/content/drive/MyDrive/GRCh38.primary_assembly.genome.fa")
OPTIMAL = ROOT / "Features/reference/codon_optimality/optimal_codons.txt"

# Prepare input required by original Step 08b
df = pd.read_csv(SOURCE)

parts = df["key"].str.extract(
    r"^(chr[^:]+):(\d+)_([ACGT]+)>([ACGT]+)$"
)

assert parts.notna().all().all(), "Unable to parse some variant keys"

step08_input = pd.DataFrame({
    "variantID": [
        f"ClinVar_{i:05d}"
        for i in range(1, len(df) + 1)
    ],
    "contig": parts[0],
    "position": parts[1].astype(int),
    "txnames": df["txnames"]
})

assert len(step08_input) == 13
step08_input.to_csv(INPUT, index=False)

# Confirm all reference files exist
for path in [SCRIPT, GTF, FASTA, OPTIMAL]:
    assert path.exists(), f"Missing: {path}"

print("=== RUNNING ORIGINAL GITHUB STEP 08b ===")

# Run in the notebook's working Python environment
original_argv = sys.argv.copy()

sys.argv = [
    str(SCRIPT),
    "--variants", str(INPUT),
    "--gtf", str(GTF),
    "--fasta", str(FASTA),
    "--optimal_codons", str(OPTIMAL),
    "--out", str(RESULT),
    "--window_nt", "100",
    "--threads", "2",
    "--aggregate", "none"
]

try:
    runpy.run_path(str(SCRIPT), run_name="__main__")
finally:
    sys.argv = original_argv

# QC
result = pd.read_csv(RESULT, sep="\t")

print("\n=== STEP 08b QC ===")
print("Output rows:", len(result))
print("Unique PTC IDs:", result["PTC_ID"].nunique())
print("\nStatus:")
print(result["status"].value_counts(dropna=False))

print("\nCodon optimality summary:")
print(result[
    [
        "CodonOptimalityFraction_CDS",
        "CodonOptimalityFraction_PTCpm100nt"
    ]
].describe())

print("\nSaved:", RESULT)

# Back up to Google Drive
BACKUP = Path(
    "/content/drive/MyDrive/NMD_ClinVar_demo_checkpoints/"
    "Features/output"
)
BACKUP.mkdir(parents=True, exist_ok=True)

shutil.copy2(INPUT, BACKUP / INPUT.name)
shutil.copy2(RESULT, BACKUP / RESULT.name)

print("Google Drive backup:", BACKUP)

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd
import shutil

root = Path("/content/NMDpredictionmodel/Features/reference/ejc")

input_bed = root / "GSE41154_EJC_hg18_unique.bed"
chain = root / "hg18ToHg38.over.chain.gz"
liftover = root / "liftOver"

output_bed = root / "GSE41154_EJC_hg38.bed"
unmapped = root / "GSE41154_EJC_unmapped.bed"

# Convert genomic coordinates
subprocess.run([
    str(liftover),
    str(input_bed),
    str(chain),
    str(output_bed),
    str(unmapped)
], check=True)

# Check conversion results
original = sum(1 for _ in open(input_bed))
converted = sum(1 for _ in open(output_bed))

print("=== EJC LIFTOVER RESULTS ===")
print("Original unique intervals:", f"{original:,}")
print("Mapped hg38 intervals:", f"{converted:,}")
print("Mapping percentage:", f"{converted/original*100:.2f}%")
print("Unmapped intervals:", f"{original-converted:,}")

print("\nOutput:", output_bed)

# Save the converted reference to Google Drive
backup = Path(
    "/content/drive/MyDrive/NMD_ClinVar_demo_checkpoints/"
    "Features/reference/ejc"
)
backup.mkdir(parents=True, exist_ok=True)

shutil.copy2(output_bed, backup / output_bed.name)

print("Google Drive backup:", backup / output_bed.name)

In [ ]:
from pathlib import Path
import runpy
import sys
import shutil
import pandas as pd

root = Path("/content/NMDpredictionmodel")
out = root / "Features/output"
backup = Path("/content/drive/MyDrive/NMD_ClinVar_demo_checkpoints/Features/output")

script = root / "Features/11_PTC_AUG_features.py"
input_file = out / "ClinVar_step11_genomicIDs.csv"
fasta = Path("/content/drive/MyDrive/GRCh38.primary_assembly.genome.fa")
gtf = root / "Annotation/reference/clinvar20_gencode_v26.gtf"
db = out / "ClinVar_PTC_AUG_transcripts.sqlite"
result = out / "ClinVar_PTC_AUG_features.tsv"

sys.argv = [
    str(script),
    "--input", str(input_file),
    "--fasta", str(fasta),
    "--gtf", str(gtf),
    "--db", str(db),
    "--output", str(result),
    "--build-db"
]

try:
    runpy.run_path(str(script), run_name="__main__")
finally:
    sys.argv = [str(script)]

# Verify output
assert result.exists(), "Step 11 output file was not created."
assert result.stat().st_size > 0, "Step 11 output is empty."

df = pd.read_csv(result, sep="\t")

print("\n=== STEP 11 RESULTS ===")
print("Output records:", len(df))
print("Columns:", df.columns.tolist())
print(df.head(13).to_string(index=False, max_colwidth=40))

# Back up results
backup.mkdir(parents=True, exist_ok=True)
shutil.copy2(result, backup / result.name)

print("\nSaved:", result)
print("Drive backup:", backup / result.name)

In [ ]:
from pathlib import Path
import runpy
import sys
import pandas as pd
import shutil

root = Path("/content/NMDpredictionmodel")
out = root / "Features/output"

script = root / "Features/11_PTC_AUG_features.py"
input_file = out / "ClinVar_step11_GENCODEmatched.csv"
fasta = Path("/content/drive/MyDrive/GRCh38.primary_assembly.genome.fa")
gtf = root / "Annotation/reference/gencode.v26.primary_assembly.annotation.gtf.gz"
db = out / "ClinVar_PTC_AUG_fullGENCODE.sqlite"
result = out / "ClinVar_PTC_AUG_GENCODEmatched.tsv"

original_argv = sys.argv[:]

try:
    sys.argv = [
        str(script),
        "--input", str(input_file),
        "--fasta", str(fasta),
        "--gtf", str(gtf),
        "--db", str(db),
        "--output", str(result)
    ]

    runpy.run_path(str(script), run_name="__main__")

finally:
    sys.argv = original_argv

df = pd.read_csv(result, sep="\t")

print("\n=== STEP 11 RESULTS ===")
print("Output records:", len(df))
print("Unique transcripts:", df["transcript_id"].nunique())

print(
    df[
        [
            "variant_id",
            "transcript_id",
            "ptc_codon_position",
            "nearest_inframe_aug_distance_nt",
            "nearest_inframe_kozak_strength"
        ]
    ].to_string(index=False)
)

if len(df) == 13:
    backup = Path(
        "/content/drive/MyDrive/NMD_ClinVar_demo_checkpoints/"
        "Features/output/ClinVar_PTC_AUG_GENCODEmatched.tsv"
    )
    backup.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(result, backup)
    print("\nAll 13 records recovered.")
    print("Drive backup:", backup)
else:
    print(f"\nOnly {len(df)} of 13 records recovered; further checking needed.")

In [ ]:
import sys
import subprocess
import pandas as pd
import shutil
from pathlib import Path

root = Path("/content/NMDpredictionmodel")
out = root / "Features/output"

script = root / "Features/13_isoform_count_features.py"
input_file = out / "ClinVar_step13_input.csv"
gtf = root / "Annotation/reference/gencode.v26.primary_assembly.annotation.gtf.gz"
result = out / "ClinVar_step13_isoform_features.csv"

# Run the original script
command = [
    sys.executable, str(script),
    "--table", str(input_file),
    "--gtf", str(gtf),
    "--output", str(result),
    "--id_col", "ensembl_gene_id"
]

subprocess.run(command, check=True)

# Inspect results
df = pd.read_csv(result)

print("\n=== STEP 13: ISOFORM COUNT FEATURES ===")
print("Output records:", len(df))
print("Missing isoform counts:", df["isoform_count"].isna().sum())

print(
    df[["ClinVar_ID", "ensembl_gene_id", "isoform_count"]]
    .to_string(index=False)
)

# Verify record count and feature values
assert len(df) == 13, "Expected 13 ClinVar records."
assert df["isoform_count"].notna().all(), "Missing isoform counts."

# Back up to Google Drive
backup = Path(
    "/content/drive/MyDrive/NMD_ClinVar_demo_checkpoints/"
    "Features/output/ClinVar_step13_isoform_features.csv"
)

backup.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(result, backup)

print("\nPASS: All 13 records have isoform counts.")
print("Saved to Google Drive:", backup)

## 7. Assemble feature matrix and conservation features

In [ ]:
from pathlib import Path

script = Path("/content/NMDpredictionmodel/Features/99_build_feature_matrix.R")
code = script.read_text()

# Keep a backup
backup = script.with_name("99_build_feature_matrix_before_merge_fix.R")
backup.write_text(code)

# Replace from base-table loading through the start of final QC
start = code.index("base_variants <- readRDS(")
end = code.index("# 11. Final QC", start)

# Include the separator immediately before the final QC heading
section_start = code.rfind(
    "# ------------------------------------------------------------------------------",
    start,
    end
)

replacement = r'''
base_variants <- readRDS(config$base_variants)

if (!"key" %in% names(base_variants)) {
    stop("Base variants must contain key.")
}

if (!"txnames" %in% names(base_variants)) {
    stop("Base variants must contain txnames.")
}

if (anyNA(base_variants$key) || anyNA(base_variants$txnames)) {
    stop("Missing key or txnames in base variants.")
}

original_rows <- nrow(base_variants)

message("Base variants: ", original_rows)
message("Unique genomic keys: ", dplyr::n_distinct(base_variants$key))

# Load the actual feature outputs.
feature_tables <- list(
    GENCODE = readRDS(config$gencode_features),
    PTBP1 = readRDS(config$ptbp1_features),
    amino_acid = readRDS(config$amino_acid_features),
    gene = readRDS(config$gene_features),
    PTC_geometry = readRDS(config$ptc_geometry_features),
    variant_scores = readRDS(config$variant_scores),
    motifs = readr::read_csv(
        config$motif_features,
        show_col_types = FALSE,
        name_repair = "minimal"
    )
)

# Transcript-level features join by txnames.
# Variant-level features join by key.
join_columns <- c(
    GENCODE = "txnames",
    PTBP1 = "txnames",
    amino_acid = "key",
    gene = "key",
    PTC_geometry = "key",
    variant_scores = "key",
    motifs = "key"
)

# Resolve repeated feature keys only when their values agree.
# Never arbitrarily choose conflicting feature annotations.
collapse_identical <- function(x, join_col, label) {

    if (!join_col %in% names(x)) {
        stop(label, " is missing joining column: ", join_col)
    }

    if (anyNA(x[[join_col]])) {
        stop(label, " contains missing joining identifiers.")
    }

    x <- as.data.frame(x)

    # Exact duplicate rows are safe to remove.
    x <- dplyr::distinct(x)

    if (anyDuplicated(x[[join_col]]) == 0) {
        return(x)
    }

    # Check whether rows sharing a join identifier have
    # conflicting values in any feature column.
    feature_cols <- setdiff(names(x), join_col)

    groups <- split(
        seq_len(nrow(x)),
        as.character(x[[join_col]])
    )

    keep <- integer(0)

    for (ids in groups) {

        if (length(ids) > 1) {

            for (column in feature_cols) {

                values <- x[[column]][ids]

                if (length(unique(values)) > 1) {
                    stop(
                        label,
                        " contains conflicting values for ",
                        join_col,
                        " = ",
                        as.character(x[[join_col]][ids[1]]),
                        " in column ",
                        column
                    )
                }
            }
        }

        keep <- c(keep, ids[1])
    }

    x[keep, , drop = FALSE]
}

feature_matrix <- base_variants

for (nm in names(feature_tables)) {

    message("\nMerging: ", nm)

    join_col <- unname(join_columns[[nm]])
    feature_table <- feature_tables[[nm]]

    # Remove non-feature identifiers from the motif matrix.
    if (nm == "motifs") {
        feature_table <- dplyr::select(
            feature_table,
            -dplyr::any_of(c("fasta_id", "txnames"))
        )
    }

    # Keep only the actual gene annotations from Module 04.
    if (nm == "gene") {
        gene_columns <- c(
            "key",
            "ensembl_gene_id",
            "hgnc_symbol",
            "pLI",
            "oe_lof_upper",
            "pLI.cat",
            "loeuf_cat",
            "MedianExpression",
            "Whole.Blood",
            "MedianExpression_log2",
            "half_life_PC1"
        )

        feature_table <- dplyr::select(
            feature_table,
            dplyr::all_of(gene_columns)
        )
    }

    # Retain only newly introduced columns.
    duplicated_columns <- setdiff(
        intersect(names(feature_table), names(feature_matrix)),
        join_col
    )

    if (length(duplicated_columns) > 0) {
        message(
            "  Existing columns skipped: ",
            paste(duplicated_columns, collapse = ", ")
        )

        feature_table <- dplyr::select(
            feature_table,
            -dplyr::all_of(duplicated_columns)
        )
    }

    feature_table <- collapse_identical(
        feature_table,
        join_col,
        nm
    )

    before <- nrow(feature_matrix)

    feature_matrix <- dplyr::left_join(
        feature_matrix,
        feature_table,
        by = join_col,
        relationship = "many-to-one"
    )

    if (nrow(feature_matrix) != before) {
        stop("Row count changed while merging ", nm)
    }

    message(
        "  Result: ",
        nrow(feature_matrix),
        " rows x ",
        ncol(feature_matrix),
        " columns"
    )
}

stopifnot(nrow(feature_matrix) == original_rows)

message("\nAll core feature merges completed.")
message("Original records preserved: ", original_rows)


# ------------------------------------------------------------------------------
'''

# Find the start of the section-11 separator and retain final QC onward.
qc_heading = code.index("# 11. Final QC", start)
qc_separator = code.rfind(
    "# ------------------------------------------------------------------------------",
    start,
    qc_heading
)

code = code[:start] + replacement + code[qc_separator:]

script.write_text(code)

print("Step 99 merge logic updated.")
print("Backup:", backup)
print("Original feature files unchanged.")
print("Ready to run for ClinVar.")

In [ ]:
subprocess.run(['Rscript','Features/99_build_feature_matrix.R'],cwd=ROOT,env={**os.environ,'NMD_DATASET':'ClinVar'},check=True)

In [ ]:
from pathlib import Path
import subprocess
import sys
import shutil
import pandas as pd

root = Path("/content/NMDpredictionmodel")

script = root / "Features/13_isoform_count_features.py"

gtf = (
    root / "Annotation/reference/"
    "gencode.v26.primary_assembly.annotation.gtf.gz"
)

input_file = (
    root / "Model/TrunCat/data/ClinVar_merged.csv"
)

output_file = (
    root / "Model/TrunCat/data/"
    "ClinVar_merged_with_isoform.csv"
)

# Verify required files
for path in [script, gtf, input_file]:
    assert path.exists(), f"Missing: {path}"

# Run existing isoform calculation
cmd = [
    sys.executable,
    str(script),
    "--table", str(input_file),
    "--gtf", str(gtf),
    "--output", str(output_file),
    "--id_col", "ensembl_gene_id"
]

print("=== CALCULATING ISOFORM COUNT ===\n")

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError("Isoform calculation failed.")

# Validate output
original = pd.read_csv(input_file, low_memory=False)
updated = pd.read_csv(output_file, low_memory=False)

assert len(updated) == 13
assert updated["ClinVar_ID"].tolist() == original["ClinVar_ID"].tolist()
assert updated["variantID"].tolist() == original["variantID"].tolist()
assert updated["isoform_count"].notna().all()

print("\n=== ISOFORM COUNTS ===")

print(
    updated[
        [
            "ClinVar_ID",
            "GENE_ID",
            "ensembl_gene_id",
            "isoform_count"
        ]
    ].to_string(index=False)
)

# Save checkpoint to Google Drive
backup_dir = (
    Path("/content/drive/MyDrive/")
    / "NMD_ClinVar_demo_checkpoints"
    / "Model/TrunCat/data"
)

backup_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(
    output_file,
    backup_dir / output_file.name
)

print("\nOriginal matrix:", original.shape)
print("Updated matrix:", updated.shape)
print("Saved:", output_file)
print("Drive backup:", backup_dir / output_file.name)

In [ ]:
from pathlib import Path
import pandas as pd

root = Path("/content/NMDpredictionmodel")

input_file = (
    root / "Model/TrunCat/data/"
    "ClinVar_merged_with_isoform.csv"
)

output_file = (
    root / "Features/output/ClinVar_conservation/"
    "ClinVar_conservation_input.csv"
)

df = pd.read_csv(input_file, low_memory=False)

assert len(df) == 13
assert "txnames_gencode" in df.columns

# Preserve the original transcript information
df["txnames_original"] = df["txnames"]

# Use transcript identifiers matching the GENCODE annotation
df["txnames"] = df["txnames_gencode"]

assert df["txnames"].notna().all()

output_file.parent.mkdir(
    parents=True,
    exist_ok=True
)

df.to_csv(output_file, index=False)

print("=== CONSERVATION TRANSCRIPT INPUT ===")

print(
    df[
        [
            "ClinVar_ID",
            "GENE_ID",
            "txnames_original",
            "txnames"
        ]
    ].to_string(index=False)
)

print("\nRows:", len(df))
print("Saved:", output_file)

In [ ]:
from pathlib import Path
import importlib.util
import pandas as pd

root = Path("/content/NMDpredictionmodel")

script = root / "Features/09_conservation_score_features.py"

input_file = (
    root / "Features/output/ClinVar_conservation/"
    "ClinVar_conservation_input.csv"
)

gtf = (
    root / "Annotation/reference/"
    "gencode.v26.primary_assembly.annotation.gtf.gz"
)

output_bed = (
    root / "Features/output/ClinVar_conservation/"
    "conservation_regions_corrected.bed"
)

for path in [script, input_file, gtf]:
    assert path.exists(), f"Missing: {path}"

# Import the existing script without running main()
spec = importlib.util.spec_from_file_location(
    "conservation_features", script
)

module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)

print("=== LOADING CLINVAR VARIANTS ===")

variants = module.load_and_normalize_variants(
    str(input_file)
)

assert len(variants) == 13

print("\n=== LOADING GENCODE ANNOTATIONS ===")

exons_by_tx, cds_bounds, _, tx_meta = (
    module.load_gtf_model_and_meta(str(gtf))
)

print("\n=== GENERATING CORRECTED CONSERVATION REGIONS ===")

rb = module.RegionBuilder(
    variants,
    exons_by_tx,
    cds_bounds,
    tx_meta,
    regions=None
)

rb.build()

bed_df = rb.to_bed(str(output_bed))

print("\n=== REGION COVERAGE ===")

bed_df["variant_index"] = (
    bed_df["name"]
    .str.extract(r"^var(\d+)_")[0]
    .astype(int)
)

bed_df["region"] = (
    bed_df["name"]
    .str.replace(r"^var\d+_", "", regex=True)
)

coverage = pd.crosstab(
    bed_df["variant_index"],
    bed_df["region"]
)

coverage = (coverage > 0).astype(int)
coverage = coverage.reindex(range(13), fill_value=0)

coverage.index = [
    f"ClinVar_{i+1:05d}"
    for i in range(13)
]

print(coverage.to_string())

print("\n=== VARIANTS PER REGION ===")
print(coverage.sum().sort_values())

print("\n=== SUMMARY ===")
print("Variants:", len(variants))
print("BED intervals:", len(bed_df))
print("Variants with transcript-based regions:",
      int((coverage.drop(columns=["ptc_100bp"], errors="ignore").sum(axis=1) > 0).sum()))
print("Saved:", output_bed)

In [ ]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import requests
import time
import shutil

root = Path("/content/NMDpredictionmodel")

outdir = (
    root / "Features/output/ClinVar_conservation"
)

bed_file = outdir / "conservation_regions_corrected.bed"

drive_dir = Path(
    "/content/drive/MyDrive/"
    "NMD_ClinVar_demo_checkpoints/"
    "Features/output/ClinVar_conservation"
)

drive_dir.mkdir(parents=True, exist_ok=True)

# Read corrected BED
bed = pd.read_csv(
    bed_file,
    sep="\t",
    header=None,
    names=[
        "chrom", "start", "end",
        "name", "score", "strand"
    ]
)

assert len(bed) == 426

tracks = {
    "phastcons": "phastCons100way",
    "phylop": "phyloP100way"
}

API = "https://api.genome.ucsc.edu/getData/track"

# Keep requests small
CHUNK_SIZE = 10000

session = requests.Session()

def fetch_scores(track, chrom, start, end):
    """Retrieve genomic conservation scores for one interval."""

    values = []

    for chunk_start in range(start, end, CHUNK_SIZE):

        chunk_end = min(chunk_start + CHUNK_SIZE, end)

        params = {
            "genome": "hg38",
            "track": track,
            "chrom": chrom,
            "start": chunk_start,
            "end": chunk_end
        }

        for attempt in range(4):

            try:
                response = session.get(
                    API,
                    params=params,
                    timeout=90
                )

                response.raise_for_status()
                data = response.json()

                records = data.get(track, [])

                # Each returned record represents a genomic span.
                # Repeat its score according to its nucleotide width.
                for record in records:

                    s = max(int(record["start"]), chunk_start)
                    e = min(int(record["end"]), chunk_end)

                    if e > s:
                        value = float(record["value"])

                        if np.isfinite(value):
                            values.extend([value] * (e - s))

                break

            except Exception as exc:

                if attempt == 3:
                    raise RuntimeError(
                        f"Failed: {track} {chrom}:"
                        f"{chunk_start}-{chunk_end}"
                    ) from exc

                time.sleep(2 ** attempt)

    return values


def calculate_track(prefix, track):

    print(f"\n=== PROCESSING {track} ===", flush=True)

    grouped_values = defaultdict(list)
    grouped_bp = defaultdict(int)

    for i, row in bed.iterrows():

        name = row["name"]

        chrom = row["chrom"]
        start = int(row["start"])
        end = int(row["end"])

        grouped_bp[name] += end - start

        scores = fetch_scores(
            track,
            chrom,
            start,
            end
        )

        grouped_values[name].extend(scores)

        if (i + 1) % 25 == 0:
            print(
                f"Processed {i+1}/{len(bed)} intervals",
                flush=True
            )

    records = []

    for name in grouped_bp:

        vals = grouped_values[name]

        records.append({
            "name": name,
            "size_bp": grouped_bp[name],
            "valid_bp": len(vals),
            "median": (
                float(np.median(vals))
                if vals else np.nan
            )
        })

    result = pd.DataFrame(records)

    output = outdir / f"{prefix}_medians.tsv"

    result.to_csv(
        output,
        sep="\t",
        index=False
    )

    shutil.copy2(
        output,
        drive_dir / output.name
    )

    print("\nCompleted:", track)
    print("Regions:", len(result))
    print(
        "Regions with scores:",
        int(result["median"].notna().sum())
    )
    print("Saved:", output)

    return result


phastcons = calculate_track(
    "phastcons",
    tracks["phastcons"]
)

phylop = calculate_track(
    "phylop",
    tracks["phylop"]
)

print("\n=== FINAL SUMMARY ===")

print(
    "PhastCons regions:",
    len(phastcons)
)

print(
    "PhyloP regions:",
    len(phylop)
)

print("All 13 ClinVar records remain represented in the BED.")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import shutil

root = Path("/content/NMDpredictionmodel")

outdir = root / "Features/output/ClinVar_conservation"

matrix_file = (
    root / "Model/TrunCat/data/ClinVar_merged_with_isoform.csv"
)

output_file = (
    root / "Model/TrunCat/data/"
    "ClinVar_merged_with_isoform_conservation.csv"
)

drive_dir = Path(
    "/content/drive/MyDrive/"
    "NMD_ClinVar_demo_checkpoints/Model/TrunCat/data"
)
drive_dir.mkdir(parents=True, exist_ok=True)

# Load existing matrix
df = pd.read_csv(matrix_file)

print("=== ORIGINAL MATRIX ===")
print("Shape:", df.shape)

assert len(df) == 13, "Expected 13 ClinVar records"

# Load completed conservation results
phast = pd.read_csv(
    outdir / "phastcons_medians.tsv",
    sep="\t"
)

phylo = pd.read_csv(
    outdir / "phylop_medians.tsv",
    sep="\t"
)

def reshape_conservation(data, prefix):

    # Example: var0_ptc_100bp -> row 0, region ptc_100bp
    extracted = data["name"].str.extract(
        r"^var(\d+)_(.+)$"
    )

    assert extracted.notna().all().all(), (
        "Unexpected conservation region name"
    )

    temp = data.copy()

    temp["row_id"] = extracted[0].astype(int)
    temp["feature"] = (
        prefix + "_" + extracted[1] + "_median"
    )

    wide = temp.pivot(
        index="row_id",
        columns="feature",
        values="median"
    )

    return wide


phast_wide = reshape_conservation(
    phast, "phastcons"
)

phylo_wide = reshape_conservation(
    phylo, "phylop"
)

conservation = phast_wide.join(
    phylo_wide,
    how="outer"
)

print("\n=== CONSERVATION FEATURES ===")
print("Feature columns:", conservation.shape[1])
print("Records:", conservation.shape[0])

assert set(conservation.index) == set(range(13)), (
    "Conservation row IDs do not match the 13 records"
)

# Preserve original row order
df = df.reset_index(drop=True)

# Update existing conservation columns if present,
# otherwise add new ones.
existing = [
    col for col in conservation.columns
    if col in df.columns
]

if existing:
    print("\nUpdating existing columns:", len(existing))
    df = df.drop(columns=existing)

df = df.join(conservation, how="left")

assert len(df) == 13

# Save new matrix without overwriting the original
df.to_csv(output_file, index=False)

# Backup to Google Drive
shutil.copy2(
    output_file,
    drive_dir / output_file.name
)

print("\n=== FINAL MATRIX ===")
print("Shape:", df.shape)
print("Conservation features:", conservation.shape[1])
print("Missing conservation values:")
print(conservation.isna().sum().to_string())

print("\nSaved:", output_file)
print("Google Drive backup:", drive_dir / output_file.name)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import shutil

root = Path("/content/NMDpredictionmodel")
out = root / "Features/output"

input_file = (
    root / "Model/TrunCat/data/"
    "ClinVar_merged_with_isoform_conservation.csv"
)

output_file = (
    root / "Model/TrunCat/data/"
    "ClinVar_merged_features_restored.csv"
)

drive_dir = Path(
    "/content/drive/MyDrive/"
    "NMD_ClinVar_demo_checkpoints/Model/TrunCat/data"
)
drive_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(input_file)

seq = pd.read_csv(
    out / "ClinVar_01b_intron_features.csv"
)

aug = pd.read_csv(
    out / "ClinVar_PTC_AUG_GENCODEmatched.tsv",
    sep="\t"
)

assert len(df) == 13
assert not seq["txnames"].duplicated().any()

print("=== RESTORING SEQUENCE FEATURES ===")

# Match sequence annotations to each ClinVar record.
# Both duplicate ClinVar records are preserved.
seq_lookup = seq.set_index("txnames")

seq_matched = seq_lookup.reindex(
    df["txnames_gencode"].astype(str)
).reset_index(drop=True)

assert len(seq_matched) == 13

seq_updated = []

for col in seq_matched.columns:

    if col in df.columns:
        old = df[col].notna().sum()
        df[col] = df[col].combine_first(seq_matched[col])
        new = df[col].notna().sum()

        if new > old:
            seq_updated.append((col, old, new))

    else:
        df[col] = seq_matched[col]
        seq_updated.append(
            (col, 0, df[col].notna().sum())
        )

print("Sequence columns added or improved:",
      len(seq_updated))

for col, old, new in seq_updated:
    print(f"{col}: {old}/13 -> {new}/13")


print("\n=== RESTORING AUG AND KOZAK FEATURES ===")

# Build matching genomic identifiers.
df_aug_id = (
    df["key"]
    .astype(str)
    .str.replace(":", "_", regex=False)
    .str.replace(">", "_", regex=False)
)

aug = aug.copy()
aug["_match_id"] = aug["variant_id"].astype(str)

# Duplicate records have identical genomic identifiers.
# Keep one annotation per identifier for the lookup.
aug_unique = aug.drop_duplicates(
    subset="_match_id",
    keep="first"
).set_index("_match_id")

aug_matched = aug_unique.reindex(
    df_aug_id
).reset_index(drop=True)

assert len(aug_matched) == 13

# Directly corresponding model features.
aug_mapping = {
    "nearest_inframe_aug_distance_nt": "aug_distance_nt",
    "nearest_inframe_kozak_strength": "kozak_strength",
    "has_plus1_frame_aug": "has_plus1_aug",
    "has_plus2_frame_aug": "has_plus2_aug"
}

aug_updated = []

for source, target in aug_mapping.items():

    if source not in aug_matched.columns:
        continue

    values = aug_matched[source]

    if target in df.columns:
        old = df[target].notna().sum()
        df[target] = df[target].combine_first(values)
    else:
        old = 0
        df[target] = values

    new = df[target].notna().sum()

    aug_updated.append((target, old, new))

for col, old, new in aug_updated:
    print(f"{col}: {old}/13 -> {new}/13")


print("\n=== FINAL QUALITY CHECK ===")

assert len(df) == 13
assert df["key"].duplicated().sum() == 1

print("Records:", len(df))
print("Columns:", len(df.columns))
print("Duplicate genomic variants:",
      df["key"].duplicated().sum())

important = [
    "cdsseqs_AU_content",
    "threeUTR_AU_content",
    "fiveUTR_AU_content",
    "fiveutrseqs.uORF",
    "threeUTR.introns",
    "fiveUTR.introns",
    "aug_distance_nt",
    "kozak_strength",
    "has_plus1_aug",
    "has_plus2_aug",
    "isoform_count",
    "phastcons_ptc_100bp_median",
    "phylop_ptc_100bp_median"
]

print("\n=== FEATURE COMPLETENESS ===")

for col in important:
    if col in df.columns:
        print(
            f"{col}: "
            f"{df[col].notna().sum()}/13"
        )

df.to_csv(output_file, index=False)

shutil.copy2(
    output_file,
    drive_dir / output_file.name
)

print("\nSaved:", output_file)
print("Google Drive backup:",
      drive_dir / output_file.name)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import shutil

root = Path("/content/NMDpredictionmodel/Model/TrunCat/data")

src = root / "ClinVar_merged_features_restored.csv"
dst = root / "ClinVar_merged_lengths_restored.csv"

df = pd.read_csv(src)

assert len(df) == 13

definitions = {
    "log2_CDS": ("cds_length", 10),
    "log2_3utr": ("threeUTR_length", 10),
    "log2_5utr": ("fiveutr_length", 7),
}

cut_names = {
    "log2_CDS": "cds_length.cut",
    "log2_3utr": "threeUTR_length.cut",
    "log2_5utr": "fiveUTR_length.cut",
}

for target, (source, cutoff) in definitions.items():

    length = pd.to_numeric(df[source], errors="coerce")

    calculated = pd.Series(
        np.where(length > 0, np.log2(length.clip(lower=1)), np.nan),
        index=df.index
    )

    if target in df.columns:
        df[target] = df[target].combine_first(calculated)
    else:
        df[target] = calculated

    cut_col = cut_names[target]
    low = f"0-{cutoff}"
    high = f">{cutoff}"

    calculated_cut = pd.Series(
        np.where(
            df[target].isna(),
            None,
            np.where(df[target] <= cutoff, low, high)
        ),
        index=df.index
    )

    if cut_col in df.columns:
        df[cut_col] = df[cut_col].combine_first(calculated_cut)
    else:
        df[cut_col] = calculated_cut

assert len(df) == 13
assert df["key"].duplicated().sum() == 1

df.to_csv(dst, index=False)

drive = Path(
    "/content/drive/MyDrive/NMD_ClinVar_demo_checkpoints/"
    "Model/TrunCat/data"
)
drive.mkdir(parents=True, exist_ok=True)
shutil.copy2(dst, drive / dst.name)

print("=== RESTORED LENGTH FEATURES ===")

for col in list(definitions) + list(cut_names.values()):
    print(f"{col}: {df[col].notna().sum()}/13")

print("\nRecords:", len(df))
print("Columns:", len(df.columns))
print("Saved:", dst)
print("Google Drive backup:", drive / dst.name)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import shutil

root = Path("/content/NMDpredictionmodel/Model/TrunCat/data")

src = root / "ClinVar_merged_newUTR_restored.csv"
if not src.exists():
    src = root / "ClinVar_merged_lengths_restored.csv"

dst = root / "ClinVar_merged_EJC_restored.csv"

df = pd.read_csv(src)

assert len(df) == 13
assert "EJC.downstream" in df.columns

ejc = pd.to_numeric(df["EJC.downstream"], errors="coerce")

df["EJC.downstream.cut"] = pd.cut(
    ejc,
    bins=[-np.inf, 2, 5, 9, 14, np.inf],
    labels=["0-2", "3-5", "6-9", "10-14", ">14"],
    right=True
)

assert len(df) == 13

df.to_csv(dst, index=False)

backup = Path(
    "/content/drive/MyDrive/NMD_ClinVar_demo_checkpoints/"
    "Model/TrunCat/data"
)
backup.mkdir(parents=True, exist_ok=True)

shutil.copy2(dst, backup / dst.name)

print("EJC.downstream.cut restored.")
print("\nCategory distribution:")
print(df["EJC.downstream.cut"].value_counts(dropna=False))

print("\nRows:", len(df))
print("Columns:", len(df.columns))
print("Saved:", dst)

## 8. Validate 619 model features and predict NMD escape

In [ ]:
import json, pandas as pd, numpy as np
from catboost import CatBoostClassifier,Pool
sys.path.insert(0,str(ROOT/'Model/TrunCat/predict'))
from predict import clean_and_align
feature_path = ROOT/'Model/TrunCat/data/ClinVar_merged_all619_features.csv'
assert feature_path.exists(), ('The full feature-generation stage has not produced '+str(feature_path)+'. Do not skip to prediction.')
features = pd.read_csv(feature_path,low_memory=False)
model = CatBoostClassifier()
model.load_model(str(ROOT/'Model/TrunCat/model/truncat.cbm'))
with open(ROOT/'Model/TrunCat/predict/training_medians.json') as f: medians=json.load(f)
X,audit=clean_and_align(features,model,medians)
assert X.shape[1]==619, f'Expected 619 features, found {X.shape[1]}'
prob=model.predict_proba(Pool(X,cat_features=model.get_cat_feature_indices()))[:,1]
THRESHOLD=0.4747
pred=features[[c for c in ['ClinVar_ID','GENE_ID','key'] if c in features.columns]].copy()
assert 'key' in pred.columns, 'Variant key is required for VCF annotation merge'
pred['TrunCat_probability_class1']=prob
pred['predicted_class']=(prob>=THRESHOLD).astype(int)
pred['predicted_label']=pred['predicted_class'].map({0:'NMD',1:'escape'})
pred['threshold_used']=THRESHOLD
out=ROOT/'Model/TrunCat/predictions'
out.mkdir(parents=True,exist_ok=True)
(out/'ClinVar_TrunCat_prediction_audit.json').write_text(json.dumps(audit,indent=2,default=str))
print('Predictions:',len(pred))


## 9. Extract ClinVar clinical significance from the same VCF

In [ ]:
import gzip
wanted=set(pred['key'].astype(str)); rows=[]
with gzip.open(VCF,'rt') as f:
    for line in f:
        if line.startswith('#'):continue
        parts=line.rstrip('\n').split('\t')
        chrom,pos,vid,ref,alts=parts[:5]
        info=parts[7]
        chrom=chrom if chrom.startswith('chr') else 'chr'+chrom
        infod=dict(x.split('=',1) for x in info.split(';') if '=' in x)
        for alt in alts.split(','):
            key=f'{chrom}:{pos}_{ref}>{alt}'
            if key in wanted:
                rows.append({'key':key,'ClinVar_VCF_ID':vid,'clnsig':infod.get('CLNSIG')})
ann=pd.DataFrame(rows)
assert not ann.empty,'No matching ClinVar VCF records'
assert ann['key'].is_unique,'Ambiguous ClinVar records: inspect duplicates before merging'
assert set(ann['key'])==wanted, 'Not every predicted variant has a ClinVar annotation'
ann['clnsig']=ann['clnsig'].replace({'.':pd.NA,'':pd.NA})
ann['clinsig_group']=ann['clnsig'].map({'Pathogenic':'PLP','Likely_pathogenic':'PLP','Uncertain_significance':'VUS'})
result=pred.merge(ann,on='key',how='left',validate='many_to_one')
assert len(result)==len(pred)
csv=out/'ClinVar_TrunCat_predictions_with_clnsig.csv'
result.to_csv(csv,index=False)
print('Saved:',csv)
print(result.to_string(index=False))


## 10. Plot predictions and save the figure

In [ ]:
import matplotlib.pyplot as plt
plot=result.drop_duplicates('key').sort_values('TrunCat_probability_class1').copy()
labels=plot['GENE_ID'].astype(str) if 'GENE_ID' in plot else plot['key'].astype(str)
fig,ax=plt.subplots(figsize=(10,max(5,0.45*len(plot))))
colors=['tab:orange' if x else 'tab:blue' for x in plot['predicted_class']]
ax.barh(labels,plot['TrunCat_probability_class1'],color=colors)
ax.axvline(THRESHOLD,color='black',linestyle='--',label=f'Threshold = {THRESHOLD}')
ax.set_xlim(0,1);ax.set_xlabel('TrunCat predicted NMD escape probability')
ax.set_title('ClinVar example: TrunCat predictions')
ax.legend();fig.tight_layout()
png=out/'ClinVar_TrunCat_prediction_plot.png'
fig.savefig(png,dpi=200,bbox_inches='tight');plt.show()
print('Saved:',png)


## Output

- `Model/TrunCat/predictions/ClinVar_TrunCat_predictions_with_clnsig.csv`
- `Model/TrunCat/predictions/ClinVar_TrunCat_prediction_audit.json`
- `Model/TrunCat/predictions/ClinVar_TrunCat_prediction_plot.png`

Clinical significance is metadata, **not** a TrunCat predictor. The 0.4747 threshold applies to class-1 (escape) probability. The example does not provide measured NMD outcomes, so no accuracy or ROC-AUC is computed.